In [1]:
# !pip install groq pandas -q

In [2]:
# Cell 2: Imports

import os
import re
import json
import time
import math
import hashlib
import getpass
import pandas as pd
from typing import Dict, Any, List
from collections import Counter
from groq import Groq

In [3]:
# Cell 3: Groq Client Setup

if not os.getenv("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter GROQ_API_KEY: ")

client = Groq(api_key=os.getenv("GROQ_API_KEY"))

MODEL = "llama-3.3-70b-versatile"

In [4]:
# Cell 4: Approx Token + Text Metrics

def approx_tokens(text: str) -> int:
    return max(1, math.ceil(len(text) / 4))


def lexical_entropy(text: str) -> float:
    words = re.findall(r"\b\w+\b", text.lower())
    if not words:
        return 0.0

    counts = Counter(words)
    total = len(words)

    return -sum((c / total) * math.log2(c / total) for c in counts.values())


def redundancy_ratio(text: str) -> float:
    units = [u.strip().lower() for u in re.split(r"[\n\.]", text) if u.strip()]
    if not units:
        return 0.0

    return 1 - (len(set(units)) / len(units))


def information_density(text: str) -> float:
    return lexical_entropy(text) / approx_tokens(text)

In [5]:
# Cell 5: Text Normalization

FILLER_TERMS = [
    "please",
    "kindly",
    "can you",
    "could you",
    "i want",
    "i need",
    "properly",
    "end to end",
    "don't miss anything",
    "as per above",
    "give me",
    "now",
    "well",
    "basically",
    "like"
]


def normalize_text(text: str) -> str:
    text = text.strip()
    text = text.replace("\r\n", "\n")
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[ \t]{2,}", " ", text)

    for term in FILLER_TERMS:
        text = re.sub(rf"\b{re.escape(term)}\b", "", text, flags=re.IGNORECASE)

    text = re.sub(r" {2,}", " ", text)
    return text.strip()

In [6]:
# Cell 6: Semantic Deduplication

def sentence_split(text: str) -> List[str]:
    parts = re.split(r"(?<=[.!?])\s+|\n+", text)
    return [p.strip() for p in parts if p.strip()]


def jaccard_similarity(a: str, b: str) -> float:
    wa = set(re.findall(r"\w+", a.lower()))
    wb = set(re.findall(r"\w+", b.lower()))

    if not wa or not wb:
        return 0.0

    return len(wa & wb) / len(wa | wb)


def semantic_deduplicate(text: str, threshold: float = 0.82) -> str:
    sentences = sentence_split(text)
    kept = []

    for sent in sentences:
        duplicate = False

        for old in kept:
            if jaccard_similarity(sent, old) >= threshold:
                duplicate = True
                break

        if not duplicate:
            kept.append(sent)

    return "\n".join(kept)

In [7]:
# Cell 7: Context Distillation

def keyword_score(sentence: str, query: str) -> float:
    q_terms = set(re.findall(r"\w+", query.lower()))
    s_terms = set(re.findall(r"\w+", sentence.lower()))

    if not s_terms:
        return 0.0

    overlap = len(q_terms & s_terms)
    density = overlap / math.sqrt(len(s_terms))

    signal_bonus = int(any(x in sentence.lower() for x in [
        "error",
        "traceback",
        "exception",
        "def ",
        "class ",
        "import ",
        "json",
        "api",
        "function",
        "bug",
        "fix",
        "summarize",
        "extract"
    ]))

    return density + signal_bonus


def distill_context(text: str, query: str, keep_ratio: float = 0.45) -> str:
    sentences = sentence_split(text)

    if len(sentences) <= 3:
        return text

    scored = [(s, keyword_score(s, query)) for s in sentences]
    scored = sorted(scored, key=lambda x: x[1], reverse=True)

    keep_n = max(3, math.ceil(len(sentences) * keep_ratio))
    selected = set(s for s, _ in scored[:keep_n])

    return "\n".join([s for s in sentences if s in selected])

In [8]:
# Cell 8: Complexity Score + Adaptive Budget

def complexity_score(query: str) -> float:
    token_len = approx_tokens(query)
    entropy = lexical_entropy(query)
    red = redundancy_ratio(query)

    code_signals = len(re.findall(
        r"```|def |class |import |error|exception|traceback|json|api|sql|bug|fix|function|summarize|extract",
        query.lower()
    ))

    length_score = min(token_len / 1200, 1.0)
    entropy_score = min(entropy / 8.0, 1.0)
    signal_score = min(code_signals / 10, 1.0)
    redundancy_penalty = red * 0.35

    score = (
        0.35 * length_score
        + 0.30 * entropy_score
        + 0.35 * signal_score
        - redundancy_penalty
    )

    return max(0.05, min(score, 1.0))


def allocate_output_budget(query: str) -> int:
    c = complexity_score(query)
    q = query.lower()

    base_min = 120
    base_max = 620

    if "summarize" in q or "summary" in q or "compress" in q:
        base_min = 90
        base_max = 220

    if "json" in q or "extract" in q:
        base_min = 120
        base_max = 300

    if "error" in q or "bug" in q or "fix" in q or "traceback" in q:
        base_min = 140
        base_max = 360

    if "linkedin" in q or "creative" in q or "post" in q:
        base_min = 180
        base_max = 450

    budget = base_min + int((base_max - base_min) * c)
    return int(budget)

In [9]:
# Cell 9: Conversation State Compression

class ConversationState:
    def __init__(self):
        self.state_summary = ""
        self.last_hash = None

    def update_state(self, user_query: str, answer: str, max_chars: int = 700):
        compressed = normalize_text(f"""
Previous user request:
{user_query}

Assistant answer:
{answer}
""")

        compressed = semantic_deduplicate(compressed)
        self.state_summary = compressed[-max_chars:]
        self.last_hash = hashlib.sha256(self.state_summary.encode()).hexdigest()[:12]

    def get_delta_prompt(self, current_query: str) -> str:
        if not self.state_summary:
            return current_query

        current_hash = hashlib.sha256(current_query.encode()).hexdigest()[:12]

        return f"""
STATE_HASH={self.last_hash}
STATE_SUMMARY:
{self.state_summary}

NEW_QUERY_HASH={current_hash}
NEW_QUERY:
{current_query}
""".strip()


memory_state = ConversationState()

In [10]:
# Cell 10: Universal Input Optimizer

def optimize_input(query: str, use_memory: bool = False) -> Dict[str, Any]:
    original = query

    q1 = normalize_text(query)
    q2 = semantic_deduplicate(q1)
    q3 = distill_context(q2, q2, keep_ratio=0.45)

    candidates = [q1, q2, q3]
    q_best = min(candidates, key=lambda x: approx_tokens(x))

    code_markers = [
        "traceback",
        "error",
        "exception",
        "code:",
        "def ",
        "class ",
        "import "
    ]

    if any(m in original.lower() for m in code_markers):
        if approx_tokens(q_best) < approx_tokens(original) * 0.45:
            q_best = q2

    if use_memory:
        q_mem = memory_state.get_delta_prompt(q_best)

        if approx_tokens(q_mem) <= approx_tokens(q_best) * 1.15:
            q_best = q_mem

    original_tokens = approx_tokens(original)
    optimized_tokens = approx_tokens(q_best)

    return {
        "original_query": original,
        "optimized_query": q_best,
        "original_est_tokens": original_tokens,
        "optimized_est_tokens": optimized_tokens,
        "estimated_input_reduction_percent": round(
            ((original_tokens - optimized_tokens) / max(original_tokens, 1)) * 100,
            2
        ),
        "complexity_score": round(complexity_score(q_best), 4),
        "adaptive_output_budget": allocate_output_budget(q_best),
        "entropy": round(lexical_entropy(q_best), 4),
        "redundancy_ratio": round(redundancy_ratio(q_best), 4),
        "information_density": round(information_density(q_best), 6),
    }

In [11]:
# Cell 11: Minimal Output Contract

def build_minimal_contract() -> str:
    return (
        "Answer compactly. "
        "No intro. No conclusion. "
        "Use only required words/code/data. "
        "Do not explain unless needed."
    )

In [12]:
# Cell 12: Groq Stop Sequences

# Groq allows maximum 4 stop sequences.
STOP_SEQUENCES = [
    "\nConclusion:",
    "\nAdditional Notes:",
    "\nReferences:",
    "\nHope this helps"
]

In [13]:
# Cell 13: Optimized Groq Call

def ask_groq_token_minimized(
    query: str,
    model: str = MODEL,
    use_memory: bool = False,
) -> Dict[str, Any]:

    optimized = optimize_input(query, use_memory=use_memory)
    max_tokens = optimized["adaptive_output_budget"]

    messages = [
        {
            "role": "system",
            "content": build_minimal_contract()
        },
        {
            "role": "user",
            "content": optimized["optimized_query"]
        }
    ]

    start = time.time()

    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0.0,
        top_p=0.68,
        max_completion_tokens=max_tokens,
        stop=STOP_SEQUENCES,
        seed=42,
    )

    answer = response.choices[0].message.content.strip()
    usage = response.usage

    result = {
        **optimized,
        "answer": answer,
        "input_tokens": usage.prompt_tokens,
        "output_tokens": usage.completion_tokens,
        "total_tokens": usage.total_tokens,
        "latency_seconds": round(time.time() - start, 3),
    }

    memory_state.update_state(query, answer)

    return result

In [14]:
# Cell 14: Baseline Groq Call

def ask_groq_baseline(query: str, model: str = MODEL) -> Dict[str, Any]:
    start = time.time()

    response = client.chat.completions.create(
        model=model,
        messages=[
            {
                "role": "user",
                "content": query
            }
        ],
        temperature=0.7,
        top_p=1.0,
        max_completion_tokens=1200,
    )

    usage = response.usage

    return {
        "answer": response.choices[0].message.content.strip(),
        "input_tokens": usage.prompt_tokens,
        "output_tokens": usage.completion_tokens,
        "total_tokens": usage.total_tokens,
        "latency_seconds": round(time.time() - start, 3),
    }

In [15]:
# Cell 15: Comparison Function

def compare(query: str, use_memory: bool = False) -> Dict[str, Any]:
    baseline = ask_groq_baseline(query)

    optimized = ask_groq_token_minimized(
        query=query,
        use_memory=use_memory
    )

    baseline_total = baseline["total_tokens"]
    optimized_total = optimized["total_tokens"]

    reduction_percent = round(
        ((baseline_total - optimized_total) / max(baseline_total, 1)) * 100,
        2
    )

    return {
        "baseline": baseline,
        "optimized": optimized,
        "reduction_percent": reduction_percent
    }


def print_compare(result: Dict[str, Any]):
    b = result["baseline"]
    o = result["optimized"]

    print("========== TOKEN RESULT ==========")
    print(f"Baseline Total Tokens   : {b['total_tokens']}")
    print(f"Optimized Total Tokens  : {o['total_tokens']}")
    print(f"Reduction %             : {result['reduction_percent']}%")

    print("\n========== INPUT TOKENS ==========")
    print(f"Baseline Input Tokens   : {b['input_tokens']}")
    print(f"Optimized Input Tokens  : {o['input_tokens']}")

    print("\n========== OUTPUT TOKENS ==========")
    print(f"Baseline Output Tokens  : {b['output_tokens']}")
    print(f"Optimized Output Tokens : {o['output_tokens']}")

    print("\n========== OPTIMIZATION METRICS ==========")
    print(f"Estimated Input Reduction % : {o['estimated_input_reduction_percent']}")
    print(f"Complexity Score            : {o['complexity_score']}")
    print(f"Adaptive Output Budget      : {o['adaptive_output_budget']}")
    print(f"Entropy                     : {o['entropy']}")
    print(f"Redundancy Ratio            : {o['redundancy_ratio']}")
    print(f"Information Density         : {o['information_density']}")

    print("\n========== OPTIMIZED ANSWER ==========\n")
    print(o["answer"])

In [16]:
# Cell 16: Test Queries

test_queries = {
    "factual_qa": """
What is Retrieval Augmented Generation? Explain with one simple example.
""",

    "code_generation": """
Write a Python function that calls Groq API and returns the answer along with
input tokens, output tokens, and total tokens.
""",

    "summarization": """
Summarize this:

ProdSync is an AI-powered SaaS platform for talent discovery, candidate evaluation,
and hiring intelligence. It helps job seekers improve employability using ATS resume
analysis, skill-gap detection, AI role recommendation, JD-resume matching, GitHub
project audit, readiness scoring, and AI mock interviews. For recruiters, it provides
semantic candidate search, AI shortlisting, automated first-round interview analysis,
and hireability reports. The platform targets Tier-2 and Tier-3 college students,
placement cells, and hiring teams in India.
""",

    "debugging": """
I am getting this Python error:

TypeError: unsupported operand type(s) for +: 'int' and 'str'

Code:
age = 25
message = "My age is " + age
print(message)

Find the bug and give the smallest correct fix.
""",

    "json_extraction": """
Extract the following information as JSON:

Candidate Name: Soumyajit Bera
Current Role: AI Engineer
Company: IBM
Experience: 2 years 9 months
Skills: Python, FastAPI, LangChain, Milvus, SQL, Machine Learning
Expected CTC: 25 LPA
Preferred Location: Kolkata
""",

    "creative": """
Write a powerful LinkedIn post announcing ProdSync as an AI-powered employability
and hiring intelligence platform for Tier-2 and Tier-3 college students in India.
Keep it founder-style, confident, and inspiring.
"""
}

In [17]:
# Cell 17: Run One Test

query = test_queries["debugging"]

result = compare(query, use_memory=False)
print_compare(result)

========== TOKEN RESULT ==========
Baseline Total Tokens   : 219
Optimized Total Tokens  : 135
Reduction %             : 38.36%

========== INPUT TOKENS ==========
Baseline Input Tokens   : 89
Optimized Input Tokens  : 102

========== OUTPUT TOKENS ==========
Baseline Output Tokens  : 130
Optimized Output Tokens : 33

========== OPTIMIZATION METRICS ==========
Estimated Input Reduction % : 17.31
Complexity Score            : 0.327
Adaptive Output Budget      : 211
Entropy                     : 4.6511
Redundancy Ratio            : 0.0
Information Density         : 0.108165

========== OPTIMIZED ANSWER ==========

**Bug:** `age` is an integer, cannot be concatenated with a string.

**Fix:** `message = "My age is " + str(age)`


In [18]:
# Cell 18: Batch Test All Use Cases

rows = []

for use_case, query in test_queries.items():
    print(f"Running: {use_case}")

    result = compare(query, use_memory=False)

    rows.append({
        "use_case": use_case,

        "baseline_total_tokens": result["baseline"]["total_tokens"],
        "optimized_total_tokens": result["optimized"]["total_tokens"],
        "reduction_percent": result["reduction_percent"],

        "baseline_input_tokens": result["baseline"]["input_tokens"],
        "optimized_input_tokens": result["optimized"]["input_tokens"],

        "baseline_output_tokens": result["baseline"]["output_tokens"],
        "optimized_output_tokens": result["optimized"]["output_tokens"],

        "complexity_score": result["optimized"]["complexity_score"],
        "adaptive_output_budget": result["optimized"]["adaptive_output_budget"],
        "estimated_input_reduction_percent": result["optimized"]["estimated_input_reduction_percent"],

        "baseline_latency_seconds": result["baseline"]["latency_seconds"],
        "optimized_latency_seconds": result["optimized"]["latency_seconds"],
    })

df = pd.DataFrame(rows)
df

Running: factual_qa
Running: code_generation
Running: summarization
Running: debugging
Running: json_extraction
Running: creative


,use_case,baseline_total_tokens,optimized_total_tokens,reduction_percent,baseline_input_tokens,optimized_input_tokens,baseline_output_tokens,optimized_output_tokens,complexity_score,adaptive_output_budget,estimated_input_reduction_percent,baseline_latency_seconds,optimized_latency_seconds
0,factual_qa,269,145,46.10,49,72,220,73,0.1298,184,5.26,1.030,0.627
1,code_generation,701,246,64.91,61,84,640,162,0.2317,235,3.12,1.606,0.648
2,summarization,225,192,14.67,146,129,79,63,0.2300,234,35.21,0.393,0.283
3,debugging,232,135,41.81,89,102,143,33,0.3270,211,17.31,0.620,0.322
4,json_extraction,207,169,18.36,104,102,103,67,0.3219,177,35.38,0.373,0.211
5,creative,575,184,68.00,77,100,498,84,0.1969,233,1.85,1.949,0.509


In [19]:
# Cell 19: Cost Estimation

INPUT_PRICE_PER_1M = 0.59
OUTPUT_PRICE_PER_1M = 0.79


def cost_usd(input_tokens: int, output_tokens: int) -> float:
    return (
        (input_tokens / 1_000_000) * INPUT_PRICE_PER_1M
        + (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_1M
    )


df["baseline_cost_usd"] = df.apply(
    lambda r: cost_usd(
        r["baseline_input_tokens"],
        r["baseline_output_tokens"]
    ),
    axis=1
)

df["optimized_cost_usd"] = df.apply(
    lambda r: cost_usd(
        r["optimized_input_tokens"],
        r["optimized_output_tokens"]
    ),
    axis=1
)

df["cost_reduction_percent"] = round(
    ((df["baseline_cost_usd"] - df["optimized_cost_usd"]) / df["baseline_cost_usd"]) * 100,
    2
)

df

,use_case,baseline_total_tokens,optimized_total_tokens,reduction_percent,baseline_input_tokens,optimized_input_tokens,baseline_output_tokens,optimized_output_tokens,complexity_score,adaptive_output_budget,estimated_input_reduction_percent,baseline_latency_seconds,optimized_latency_seconds,baseline_cost_usd,optimized_cost_usd,cost_reduction_percent
0,factual_qa,269,145,46.10,49,72,220,73,0.1298,184,5.26,1.030,0.627,0.000203,0.000100,50.59
1,code_generation,701,246,64.91,61,84,640,162,0.2317,235,3.12,1.606,0.648,0.000542,0.000178,67.22
2,summarization,225,192,14.67,146,129,79,63,0.2300,234,35.21,0.393,0.283,0.000149,0.000126,15.26
3,debugging,232,135,41.81,89,102,143,33,0.3270,211,17.31,0.620,0.322,0.000165,0.000086,47.88
4,json_extraction,207,169,18.36,104,102,103,67,0.3219,177,35.38,0.373,0.211,0.000143,0.000113,20.75
5,creative,575,184,68.00,77,100,498,84,0.1969,233,1.85,1.949,0.509,0.000439,0.000125,71.43


In [20]:
# Cell 20: Average Reduction Summary

summary = {
    "avg_token_reduction_percent": round(df["reduction_percent"].mean(), 2),
    "avg_cost_reduction_percent": round(df["cost_reduction_percent"].mean(), 2),
    "best_use_case": df.sort_values("reduction_percent", ascending=False).iloc[0]["use_case"],
    "worst_use_case": df.sort_values("reduction_percent", ascending=True).iloc[0]["use_case"],
}

summary

{'avg_token_reduction_percent': np.float64(42.31),
 'avg_cost_reduction_percent': np.float64(45.52),
 'best_use_case': 'creative',
 'worst_use_case': 'summarization'}

In [21]:
# Cell 21: Inspect Optimized Prompt

sample_query = test_queries["summarization"]

optimized_debug = optimize_input(sample_query)

print("Original Estimated Tokens:", optimized_debug["original_est_tokens"])
print("Optimized Estimated Tokens:", optimized_debug["optimized_est_tokens"])
print("Estimated Reduction %:", optimized_debug["estimated_input_reduction_percent"])

print("\n========== OPTIMIZED QUERY ==========\n")
print(optimized_debug["optimized_query"])

Original Estimated Tokens: 142
Optimized Estimated Tokens: 92
Estimated Reduction %: 35.21

========== OPTIMIZED QUERY ==========

ProdSync is an AI-powered SaaS platform for talent discovery, candidate evaluation,
It helps job seekers improve employability using ATS resume
analysis, skill-gap detection, AI role recommendation, JD-resume matching, GitHub
semantic candidate search, AI shortlisting, automated first-round interview analysis,
The platform targets Tier-2 and Tier-3 college students,


In [22]:
# Cell 22: Token Efficiency Score

def token_efficiency_score(
    information_density_value: float,
    quality_score: float,
    total_tokens: int
) -> float:
    """
    TES = (Information Density × Quality Score) / Total Tokens

    Higher TES = better efficiency.
    """

    if total_tokens <= 0:
        return 0.0

    return round(
        (information_density_value * quality_score) / total_tokens,
        8
    )

In [23]:
# Cell 23: Lightweight Quality Score Estimator

def estimate_quality_score(answer: str, query: str) -> float:
    """
    Non-LLM quality approximation.
    Score range: 0 to 1.

    Uses:
    - query-answer term overlap
    - answer completeness
    - penalty for too-short answers
    - penalty for repetition
    """

    query_terms = set(re.findall(r"\w+", query.lower()))
    answer_terms = set(re.findall(r"\w+", answer.lower()))

    if not answer_terms:
        return 0.0

    overlap_score = len(query_terms & answer_terms) / max(len(query_terms), 1)

    answer_token_count = approx_tokens(answer)

    length_score = min(answer_token_count / 120, 1.0)

    repetition_penalty = redundancy_ratio(answer)

    quality = (
        0.50 * overlap_score
        + 0.35 * length_score
        + 0.15 * (1 - repetition_penalty)
    )

    return round(max(0.0, min(quality, 1.0)), 4)

In [24]:
# Cell 24: Add TES to Single Comparison

def compare_with_tes(query: str, use_memory: bool = False) -> Dict[str, Any]:
    result = compare(query, use_memory=use_memory)

    baseline_answer = result["baseline"]["answer"]
    optimized_answer = result["optimized"]["answer"]

    baseline_quality = estimate_quality_score(baseline_answer, query)
    optimized_quality = estimate_quality_score(optimized_answer, query)

    baseline_density = information_density(baseline_answer)
    optimized_density = result["optimized"]["information_density"]

    baseline_tes = token_efficiency_score(
        information_density_value=baseline_density,
        quality_score=baseline_quality,
        total_tokens=result["baseline"]["total_tokens"]
    )

    optimized_tes = token_efficiency_score(
        information_density_value=optimized_density,
        quality_score=optimized_quality,
        total_tokens=result["optimized"]["total_tokens"]
    )

    tes_improvement = round(
        ((optimized_tes - baseline_tes) / max(baseline_tes, 1e-8)) * 100,
        2
    )

    result["baseline"]["quality_score"] = baseline_quality
    result["optimized"]["quality_score"] = optimized_quality

    result["baseline"]["token_efficiency_score"] = baseline_tes
    result["optimized"]["token_efficiency_score"] = optimized_tes

    result["tes_improvement_percent"] = tes_improvement

    return result

In [25]:
# Cell 25: Print TES Comparison

def print_compare_with_tes(result: Dict[str, Any]):
    print_compare(result)

    print("\n========== TOKEN EFFICIENCY SCORE ==========")
    print(f"Baseline Quality Score  : {result['baseline']['quality_score']}")
    print(f"Optimized Quality Score : {result['optimized']['quality_score']}")

    print(f"Baseline TES            : {result['baseline']['token_efficiency_score']}")
    print(f"Optimized TES           : {result['optimized']['token_efficiency_score']}")
    print(f"TES Improvement %       : {result['tes_improvement_percent']}%")

In [26]:
# Cell 26: Run One TES Test

query = test_queries["debugging"]

tes_result = compare_with_tes(query, use_memory=False)
print_compare_with_tes(tes_result)

========== TOKEN RESULT ==========
Baseline Total Tokens   : 213
Optimized Total Tokens  : 135
Reduction %             : 36.62%

========== INPUT TOKENS ==========
Baseline Input Tokens   : 89
Optimized Input Tokens  : 102

========== OUTPUT TOKENS ==========
Baseline Output Tokens  : 124
Optimized Output Tokens : 33

========== OPTIMIZATION METRICS ==========
Estimated Input Reduction % : 17.31
Complexity Score            : 0.327
Adaptive Output Budget      : 211
Entropy                     : 4.6511
Redundancy Ratio            : 0.0
Information Density         : 0.108165

========== OPTIMIZED ANSWER ==========

**Bug:** `age` is an integer, cannot be concatenated with a string.

**Fix:** `message = "My age is " + str(age)`

========== TOKEN EFFICIENCY SCORE ==========
Baseline Quality Score  : 0.7158
Optimized Quality Score : 0.3553
Baseline TES            : 0.00014409
Optimized TES           : 0.00028467
TES Improvement %       : 97.56%


In [27]:
# Cell 27: Batch TES Evaluation

tes_rows = []

for use_case, query in test_queries.items():
    print(f"Running TES evaluation: {use_case}")

    result = compare_with_tes(query, use_memory=False)

    tes_rows.append({
        "use_case": use_case,

        "baseline_total_tokens": result["baseline"]["total_tokens"],
        "optimized_total_tokens": result["optimized"]["total_tokens"],
        "token_reduction_percent": result["reduction_percent"],

        "baseline_quality_score": result["baseline"]["quality_score"],
        "optimized_quality_score": result["optimized"]["quality_score"],

        "baseline_tes": result["baseline"]["token_efficiency_score"],
        "optimized_tes": result["optimized"]["token_efficiency_score"],
        "tes_improvement_percent": result["tes_improvement_percent"],

        "optimized_information_density": result["optimized"]["information_density"],
        "complexity_score": result["optimized"]["complexity_score"],
        "adaptive_output_budget": result["optimized"]["adaptive_output_budget"],
    })

tes_df = pd.DataFrame(tes_rows)
tes_df

Running TES evaluation: factual_qa
Running TES evaluation: code_generation
Running TES evaluation: summarization
Running TES evaluation: debugging
Running TES evaluation: json_extraction
Running TES evaluation: creative


,use_case,baseline_total_tokens,optimized_total_tokens,token_reduction_percent,baseline_quality_score,optimized_quality_score,baseline_tes,optimized_tes,tes_improvement_percent,optimized_information_density,complexity_score,adaptive_output_budget
0,factual_qa,411,148,63.99,0.8917,0.6183,0.000031,0.000771,2405.69,0.184552,0.1298,184
1,code_generation,720,246,65.83,0.9642,0.6000,0.000012,0.000320,2670.50,0.131311,0.2317,235
2,summarization,206,192,6.80,0.6314,0.5427,0.000189,0.000166,-11.89,0.058877,0.2300,234
3,debugging,203,135,33.50,0.6202,0.3553,0.000144,0.000285,98.09,0.108165,0.3270,211
4,json_extraction,207,169,18.36,0.6999,0.5487,0.000204,0.000350,71.83,0.107704,0.3219,177
5,creative,581,182,68.67,0.8167,0.6967,0.000016,0.000350,2133.61,0.091315,0.1969,233


In [28]:
# Cell 28: TES Summary

tes_summary = {
    "avg_token_reduction_percent": round(tes_df["token_reduction_percent"].mean(), 2),
    "avg_tes_improvement_percent": round(tes_df["tes_improvement_percent"].mean(), 2),
    "best_tes_use_case": tes_df.sort_values("tes_improvement_percent", ascending=False).iloc[0]["use_case"],
    "worst_tes_use_case": tes_df.sort_values("tes_improvement_percent", ascending=True).iloc[0]["use_case"],
}

tes_summary

{'avg_token_reduction_percent': np.float64(42.86),
 'avg_tes_improvement_percent': np.float64(1227.97),
 'best_tes_use_case': 'code_generation',
 'worst_tes_use_case': 'summarization'}

In [29]:
# Cell 29: Advanced Optimization Metrics

def compression_factor(original_tokens: int, optimized_tokens: int) -> float:
    """
    CF = T_original / T_optimized

    Higher is better.
    CF = 1 means no compression.
    """

    if optimized_tokens <= 0:
        return 1.0

    return round(original_tokens / optimized_tokens, 4)


def cost_efficiency_score(
    information_density_value: float,
    quality_score: float,
    cost_value: float
) -> float:
    """
    CTES = (Information Density × Quality Score) / Cost

    Higher is better.
    """

    if cost_value <= 0:
        return 0.0

    return round(
        (information_density_value * quality_score) / cost_value,
        8
    )


def weighted_token_efficiency_score(
    information_density_value: float,
    quality_score: float,
    complexity_value: float,
    total_tokens: int
) -> float:
    """
    WTES = (Information Density × Quality Score × Complexity) / Total Tokens

    Higher is better.
    """

    if total_tokens <= 0:
        return 0.0

    return round(
        (information_density_value * quality_score * complexity_value) / total_tokens,
        8
    )


def advanced_token_efficiency_score(
    information_density_value: float,
    quality_score: float,
    complexity_value: float,
    compression_factor_value: float,
    total_tokens: int
) -> float:
    """
    ATES = (Information Density × Quality Score × Complexity × Compression Factor) / Total Tokens

    Higher is better.
    """

    if total_tokens <= 0:
        return 0.0

    return round(
        (
            information_density_value
            * quality_score
            * complexity_value
            * compression_factor_value
        ) / total_tokens,
        8
    )

In [30]:
# Cell 29: Advanced Optimization Metrics

def compression_factor(original_tokens: int, optimized_tokens: int) -> float:
    """
    CF = T_original / T_optimized

    Higher is better.
    CF = 1 means no compression.
    """

    if optimized_tokens <= 0:
        return 1.0

    return round(original_tokens / optimized_tokens, 4)


def cost_efficiency_score(
    information_density_value: float,
    quality_score: float,
    cost_value: float
) -> float:
    """
    CTES = (Information Density × Quality Score) / Cost

    Higher is better.
    """

    if cost_value <= 0:
        return 0.0

    return round(
        (information_density_value * quality_score) / cost_value,
        8
    )


def weighted_token_efficiency_score(
    information_density_value: float,
    quality_score: float,
    complexity_value: float,
    total_tokens: int
) -> float:
    """
    WTES = (Information Density × Quality Score × Complexity) / Total Tokens

    Higher is better.
    """

    if total_tokens <= 0:
        return 0.0

    return round(
        (information_density_value * quality_score * complexity_value) / total_tokens,
        8
    )


def advanced_token_efficiency_score(
    information_density_value: float,
    quality_score: float,
    complexity_value: float,
    compression_factor_value: float,
    total_tokens: int
) -> float:
    """
    ATES = (Information Density × Quality Score × Complexity × Compression Factor) / Total Tokens

    Higher is better.
    """

    if total_tokens <= 0:
        return 0.0

    return round(
        (
            information_density_value
            * quality_score
            * complexity_value
            * compression_factor_value
        ) / total_tokens,
        8
    )

In [31]:
# Cell 31: Full Advanced Evaluation for One Query

def compare_with_advanced_metrics(query: str, use_memory: bool = False) -> Dict[str, Any]:
    result = compare_with_tes(query, use_memory=use_memory)

    b = result["baseline"]
    o = result["optimized"]

    baseline_cost = cost_usd(
        b["input_tokens"],
        b["output_tokens"]
    )

    optimized_cost = cost_usd(
        o["input_tokens"],
        o["output_tokens"]
    )

    cost_reduction_percent = round(
        ((baseline_cost - optimized_cost) / max(baseline_cost, 1e-12)) * 100,
        2
    )

    cf = compression_factor(
        o["original_est_tokens"],
        o["optimized_est_tokens"]
    )

    optimized_ctes = cost_efficiency_score(
        information_density_value=o["information_density"],
        quality_score=o["quality_score"],
        cost_value=optimized_cost
    )

    optimized_wtes = weighted_token_efficiency_score(
        information_density_value=o["information_density"],
        quality_score=o["quality_score"],
        complexity_value=o["complexity_score"],
        total_tokens=o["total_tokens"]
    )

    optimized_ates = advanced_token_efficiency_score(
        information_density_value=o["information_density"],
        quality_score=o["quality_score"],
        complexity_value=o["complexity_score"],
        compression_factor_value=cf,
        total_tokens=o["total_tokens"]
    )

    result["baseline"]["cost_usd"] = round(baseline_cost, 8)
    result["optimized"]["cost_usd"] = round(optimized_cost, 8)

    result["optimized"]["compression_factor"] = cf
    result["optimized"]["ctes"] = optimized_ctes
    result["optimized"]["wtes"] = optimized_wtes
    result["optimized"]["ates"] = optimized_ates
    result["cost_reduction_percent"] = cost_reduction_percent

    return result

In [32]:
# Cell 32: Print Advanced Metrics

def print_advanced_metrics(result: Dict[str, Any]):
    print_compare_with_tes(result)

    print("\n========== ADVANCED OPTIMIZATION METRICS ==========")
    print(f"Baseline Cost USD        : {result['baseline']['cost_usd']}")
    print(f"Optimized Cost USD       : {result['optimized']['cost_usd']}")
    print(f"Cost Reduction %         : {result['cost_reduction_percent']}")

    print(f"Compression Factor       : {result['optimized']['compression_factor']}")
    print(f"CTES                     : {result['optimized']['ctes']}")
    print(f"WTES                     : {result['optimized']['wtes']}")
    print(f"ATES                     : {result['optimized']['ates']}")

In [33]:
# Cell 33: Run One Advanced Test

query = test_queries["debugging"]

advanced_result = compare_with_advanced_metrics(query, use_memory=False)
print_advanced_metrics(advanced_result)

========== TOKEN RESULT ==========
Baseline Total Tokens   : 199
Optimized Total Tokens  : 135
Reduction %             : 32.16%

========== INPUT TOKENS ==========
Baseline Input Tokens   : 89
Optimized Input Tokens  : 102

========== OUTPUT TOKENS ==========
Baseline Output Tokens  : 110
Optimized Output Tokens : 33

========== OPTIMIZATION METRICS ==========
Estimated Input Reduction % : 17.31
Complexity Score            : 0.327
Adaptive Output Budget      : 211
Entropy                     : 4.6511
Redundancy Ratio            : 0.0
Information Density         : 0.108165

========== OPTIMIZED ANSWER ==========

**Bug:** `age` is an integer, cannot be concatenated with a string.

**Fix:** `message = "My age is " + str(age)`

========== TOKEN EFFICIENCY SCORE ==========
Baseline Quality Score  : 0.6544
Optimized Quality Score : 0.3553
Baseline TES            : 0.00015485
Optimized TES           : 0.00028467
TES Improvement %       : 83.84%

========== ADVANCED OPTIMIZATION METRICS =====

In [34]:
# Cell 34: Batch Advanced Evaluation

advanced_rows = []
for_use_cases = test_queries.items()

for use_case, query in for_use_cases:
    print(f"Running advanced evaluation: {use_case}")

    result = compare_with_advanced_metrics(query, use_memory=False)

    advanced_rows.append({
        "use_case": use_case,

        "baseline_total_tokens": result["baseline"]["total_tokens"],
        "optimized_total_tokens": result["optimized"]["total_tokens"],
        "token_reduction_percent": result["reduction_percent"],

        "baseline_quality_score": result["baseline"]["quality_score"],
        "optimized_quality_score": result["optimized"]["quality_score"],

        "baseline_tes": result["baseline"]["token_efficiency_score"],
        "optimized_tes": result["optimized"]["token_efficiency_score"],
        "tes_improvement_percent": result["tes_improvement_percent"],

        "baseline_cost_usd": result["baseline"]["cost_usd"],
        "optimized_cost_usd": result["optimized"]["cost_usd"],
        "cost_reduction_percent": result["cost_reduction_percent"],

        "compression_factor": result["optimized"]["compression_factor"],
        "complexity_score": result["optimized"]["complexity_score"],
        "information_density": result["optimized"]["information_density"],

        "ctes": result["optimized"]["ctes"],
        "wtes": result["optimized"]["wtes"],
        "ates": result["optimized"]["ates"],
    })


advanced_df = pd.DataFrame(advanced_rows)
advanced_df

Running advanced evaluation: factual_qa
Running advanced evaluation: code_generation
Running advanced evaluation: summarization
Running advanced evaluation: debugging
Running advanced evaluation: json_extraction
Running advanced evaluation: creative


,use_case,baseline_total_tokens,optimized_total_tokens,token_reduction_percent,baseline_quality_score,optimized_quality_score,baseline_tes,optimized_tes,tes_improvement_percent,baseline_cost_usd,optimized_cost_usd,cost_reduction_percent,compression_factor,complexity_score,information_density,ctes,wtes,ates
0,factual_qa,376,148,60.64,0.8917,0.6183,0.000036,0.000771,2015.23,0.000287,0.000103,64.31,1.0556,0.1298,0.184552,1113.036496,0.000100,0.000106
1,code_generation,677,261,61.45,0.9621,0.5961,0.000013,0.000300,2212.26,0.000523,0.000189,63.76,1.0323,0.2317,0.131311,413.297888,0.000069,0.000072
2,summarization,218,192,11.93,0.7005,0.5427,0.000180,0.000166,-7.43,0.000143,0.000126,11.98,1.5435,0.2300,0.058877,253.833396,0.000038,0.000059
3,debugging,211,135,36.02,0.6927,0.3553,0.000145,0.000285,96.60,0.000149,0.000086,42.07,1.2093,0.3270,0.108165,445.577096,0.000093,0.000113
4,json_extraction,216,169,21.76,0.9058,0.5487,0.000215,0.000350,62.50,0.000150,0.000113,24.51,1.5476,0.3219,0.107704,522.475332,0.000113,0.000174
5,creative,487,184,62.22,0.8333,0.7025,0.000022,0.000349,1476.80,0.000369,0.000125,66.06,1.0189,0.1969,0.091315,511.716556,0.000069,0.000070


In [35]:
# Cell 35: Self-Contained GOES Calculation

def normalize_metric(value: float, min_value: float, max_value: float) -> float:
    """
    Min-max normalization into [0, 1].
    """

    if max_value == min_value:
        return 0.0

    return round(
        (value - min_value) / (max_value - min_value),
        6
    )


def global_optimization_efficiency_score(
    tes_norm: float,
    cost_reduction_norm: float,
    quality_score: float,
    compression_norm: float,
    weights: Dict[str, float] = None
) -> float:
    """
    GOES =
        0.35 × TES_norm
      + 0.25 × CostReduction_norm
      + 0.20 × Quality
      + 0.20 × Compression_norm
    """

    if weights is None:
        weights = {
            "tes": 0.35,
            "cost": 0.25,
            "quality": 0.20,
            "compression": 0.20
        }

    score = (
        weights["tes"] * tes_norm
        + weights["cost"] * cost_reduction_norm
        + weights["quality"] * quality_score
        + weights["compression"] * compression_norm
    )

    return round(score, 6)


def add_goes_to_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    """
    Adds normalized metrics and GOES to advanced_df.
    This function is self-contained and safe to rerun.
    """

    required_columns = [
        "optimized_tes",
        "cost_reduction_percent",
        "compression_factor",
        "optimized_quality_score"
    ]

    missing = [col for col in required_columns if col not in df.columns]

    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    df = df.copy()

    tes_min = df["optimized_tes"].min()
    tes_max = df["optimized_tes"].max()

    cost_min = df["cost_reduction_percent"].min()
    cost_max = df["cost_reduction_percent"].max()

    compression_min = df["compression_factor"].min()
    compression_max = df["compression_factor"].max()

    df["tes_norm"] = df["optimized_tes"].apply(
        lambda x: normalize_metric(x, tes_min, tes_max)
    )

    df["cost_reduction_norm"] = df["cost_reduction_percent"].apply(
        lambda x: normalize_metric(x, cost_min, cost_max)
    )

    df["compression_norm"] = df["compression_factor"].apply(
        lambda x: normalize_metric(x, compression_min, compression_max)
    )

    df["goes"] = df.apply(
        lambda r: global_optimization_efficiency_score(
            tes_norm=r["tes_norm"],
            cost_reduction_norm=r["cost_reduction_norm"],
            quality_score=r["optimized_quality_score"],
            compression_norm=r["compression_norm"]
        ),
        axis=1
    )

    return df


advanced_df = add_goes_to_dataframe(advanced_df)
advanced_df

,use_case,baseline_total_tokens,optimized_total_tokens,token_reduction_percent,baseline_quality_score,optimized_quality_score,baseline_tes,optimized_tes,tes_improvement_percent,baseline_cost_usd,...,compression_factor,complexity_score,information_density,ctes,wtes,ates,tes_norm,cost_reduction_norm,compression_norm,goes
0,factual_qa,376,148,60.64,0.8917,0.6183,0.000036,0.000771,2015.23,0.000287,...,1.0556,0.1298,0.184552,1113.036496,0.000100,0.000106,1.000000,0.967641,0.069416,0.729453
1,code_generation,677,261,61.45,0.9621,0.5961,0.000013,0.000300,2212.26,0.000523,...,1.0323,0.2317,0.131311,413.297888,0.000069,0.000072,0.220781,0.957470,0.025345,0.440930
2,summarization,218,192,11.93,0.7005,0.5427,0.000180,0.000166,-7.43,0.000143,...,1.5435,0.2300,0.058877,253.833396,0.000038,0.000059,0.000000,0.000000,0.992245,0.306989
3,debugging,211,135,36.02,0.6927,0.3553,0.000145,0.000285,96.60,0.000149,...,1.2093,0.3270,0.108165,445.577096,0.000093,0.000113,0.195590,0.556398,0.360129,0.350642
4,json_extraction,216,169,21.76,0.9058,0.5487,0.000215,0.000350,62.50,0.000150,...,1.5476,0.3219,0.107704,522.475332,0.000113,0.000174,0.303136,0.231694,1.000000,0.473761
5,creative,487,184,62.22,0.8333,0.7025,0.000022,0.000349,1476.80,0.000369,...,1.0189,0.1969,0.091315,511.716556,0.000069,0.000070,0.301383,1.000000,0.000000,0.495984


In [36]:
# Cell 35: Compute GOES Across Batch

def add_goes_to_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    df["tes_norm"] = df["optimized_tes"].apply(
        lambda x: normalize_metric(
            x,
            df["optimized_tes"].min(),
            df["optimized_tes"].max()
        )
    )

    df["cost_reduction_norm"] = df["cost_reduction_percent"].apply(
        lambda x: normalize_metric(
            x,
            df["cost_reduction_percent"].min(),
            df["cost_reduction_percent"].max()
        )
    )

    df["compression_norm"] = df["compression_factor"].apply(
        lambda x: normalize_metric(
            x,
            df["compression_factor"].min(),
            df["compression_factor"].max()
        )
    )

    df["goes"] = df.apply(
        lambda r: global_optimization_efficiency_score(
            tes_norm=r["tes_norm"],
            cost_reduction_norm=r["cost_reduction_norm"],
            quality_score=r["optimized_quality_score"],
            compression_norm=r["compression_norm"]
        ),
        axis=1
    )

    return df


advanced_df = add_goes_to_dataframe(advanced_df)
advanced_df

,use_case,baseline_total_tokens,optimized_total_tokens,token_reduction_percent,baseline_quality_score,optimized_quality_score,baseline_tes,optimized_tes,tes_improvement_percent,baseline_cost_usd,...,compression_factor,complexity_score,information_density,ctes,wtes,ates,tes_norm,cost_reduction_norm,compression_norm,goes
0,factual_qa,376,148,60.64,0.8917,0.6183,0.000036,0.000771,2015.23,0.000287,...,1.0556,0.1298,0.184552,1113.036496,0.000100,0.000106,1.000000,0.967641,0.069416,0.729453
1,code_generation,677,261,61.45,0.9621,0.5961,0.000013,0.000300,2212.26,0.000523,...,1.0323,0.2317,0.131311,413.297888,0.000069,0.000072,0.220781,0.957470,0.025345,0.440930
2,summarization,218,192,11.93,0.7005,0.5427,0.000180,0.000166,-7.43,0.000143,...,1.5435,0.2300,0.058877,253.833396,0.000038,0.000059,0.000000,0.000000,0.992245,0.306989
3,debugging,211,135,36.02,0.6927,0.3553,0.000145,0.000285,96.60,0.000149,...,1.2093,0.3270,0.108165,445.577096,0.000093,0.000113,0.195590,0.556398,0.360129,0.350642
4,json_extraction,216,169,21.76,0.9058,0.5487,0.000215,0.000350,62.50,0.000150,...,1.5476,0.3219,0.107704,522.475332,0.000113,0.000174,0.303136,0.231694,1.000000,0.473761
5,creative,487,184,62.22,0.8333,0.7025,0.000022,0.000349,1476.80,0.000369,...,1.0189,0.1969,0.091315,511.716556,0.000069,0.000070,0.301383,1.000000,0.000000,0.495984


In [37]:
 # Cell 36: GOES Summary

goes_summary = {
    "avg_token_reduction_percent": round(advanced_df["token_reduction_percent"].mean(), 2),
    "avg_cost_reduction_percent": round(advanced_df["cost_reduction_percent"].mean(), 2),
    "avg_tes_improvement_percent": round(advanced_df["tes_improvement_percent"].mean(), 2),
    "avg_goes": round(advanced_df["goes"].mean(), 4),

    "best_goes_use_case": advanced_df.sort_values("goes", ascending=False).iloc[0]["use_case"],
    "worst_goes_use_case": advanced_df.sort_values("goes", ascending=True).iloc[0]["use_case"],

    "best_token_reduction_use_case": advanced_df.sort_values("token_reduction_percent", ascending=False).iloc[0]["use_case"],
    "worst_token_reduction_use_case": advanced_df.sort_values("token_reduction_percent", ascending=True).iloc[0]["use_case"],
}

goes_summary

{'avg_token_reduction_percent': np.float64(42.34),
 'avg_cost_reduction_percent': np.float64(45.45),
 'avg_tes_improvement_percent': np.float64(975.99),
 'avg_goes': np.float64(0.4663),
 'best_goes_use_case': 'factual_qa',
 'worst_goes_use_case': 'summarization',
 'best_token_reduction_use_case': 'creative',
 'worst_token_reduction_use_case': 'summarization'}

In [38]:
# Cell 37: Rank Use Cases by GOES

advanced_df.sort_values("goes", ascending=False)[
    [
        "use_case",
        "goes",
        "token_reduction_percent",
        "cost_reduction_percent",
        "optimized_quality_score",
        "compression_factor",
        "optimized_tes",
        "ctes",
        "wtes",
        "ates"
    ]
]

,use_case,goes,token_reduction_percent,cost_reduction_percent,optimized_quality_score,compression_factor,optimized_tes,ctes,wtes,ates
0,factual_qa,0.729453,60.64,64.31,0.6183,1.0556,0.000771,1113.036496,0.000100,0.000106
5,creative,0.495984,62.22,66.06,0.7025,1.0189,0.000349,511.716556,0.000069,0.000070
4,json_extraction,0.473761,21.76,24.51,0.5487,1.5476,0.000350,522.475332,0.000113,0.000174
1,code_generation,0.440930,61.45,63.76,0.5961,1.0323,0.000300,413.297888,0.000069,0.000072
3,debugging,0.350642,36.02,42.07,0.3553,1.2093,0.000285,445.577096,0.000093,0.000113
2,summarization,0.306989,11.93,11.98,0.5427,1.5435,0.000166,253.833396,0.000038,0.000059


In [39]:
# Cell 38: Install Advanced Optimization Dependencies

!pip3 install sentence-transformers transformers torch scikit-learn -q


[notice] A new release of pip is available: 26.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [40]:
# Cell 39: Imports for Advanced Optimizer

import numpy as np
import torch

from sentence_transformers import SentenceTransformer
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from transformers import AutoTokenizer, AutoModelForCausalLM

/Users/soumyajitbera/Documents/GitHub/Token_monitoring_and_optimisation/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [41]:
# Cell 40: Load Local Embedding + Perplexity Models

EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
PPL_MODEL_NAME = "distilgpt2"

embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

ppl_tokenizer = AutoTokenizer.from_pretrained(PPL_MODEL_NAME)
ppl_model = AutoModelForCausalLM.from_pretrained(PPL_MODEL_NAME)
ppl_model.eval()

Loading weights: 100%|██████████| 76/76 [00:00<00:00, 15061.76it/s]


GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-5): 6 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [42]:
# Cell 41: Embedding-Based Semantic Pruning

def cosine_similarity_np(a: np.ndarray, b: np.ndarray) -> float:
    denom = np.linalg.norm(a) * np.linalg.norm(b)

    if denom == 0:
        return 0.0

    return float(np.dot(a, b) / denom)


def embedding_semantic_prune(
    text: str,
    query: str,
    similarity_threshold: float = 0.92
) -> str:
    """
    Replaces Jaccard-based deduplication.
    Uses dense embeddings to remove semantically redundant spans.
    """

    spans = sentence_split(text)

    if len(spans) <= 2:
        return text

    span_embeddings = embedding_model.encode(spans, convert_to_numpy=True)
    query_embedding = embedding_model.encode([query], convert_to_numpy=True)[0]

    kept_spans = []
    kept_embeddings = []

    for span, emb in zip(spans, span_embeddings):
        duplicate = False

        for kept_emb in kept_embeddings:
            sim = cosine_similarity_np(emb, kept_emb)

            if sim >= similarity_threshold:
                duplicate = True
                break

        if not duplicate:
            kept_spans.append(span)
            kept_embeddings.append(emb)

    # Re-rank retained spans by query relevance
    ranked = []

    for span, emb in zip(kept_spans, kept_embeddings):
        relevance = cosine_similarity_np(emb, query_embedding)
        ranked.append((span, relevance))

    # Preserve original order but remove very low relevance spans only for long inputs
    if len(kept_spans) > 6:
        relevance_scores = [r for _, r in ranked]
        cutoff = np.percentile(relevance_scores, 25)

        selected = {
            span
            for span, relevance in ranked
            if relevance >= cutoff
        }

        kept_spans = [span for span in kept_spans if span in selected]

    return "\n".join(kept_spans)

In [43]:
# Cell 42: Perplexity-Based Complexity Signal

def calculate_perplexity(text: str, max_length: int = 512) -> float:
    """
    Computes approximate perplexity using a small local causal LM.
    Higher PPL generally means more complex/unusual text.
    """

    if not text.strip():
        return 1.0

    inputs = ppl_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=max_length
    )

    with torch.no_grad():
        outputs = ppl_model(
            **inputs,
            labels=inputs["input_ids"]
        )

    loss = outputs.loss.item()
    ppl = math.exp(min(loss, 20))

    return round(float(ppl), 4)


def structural_signal_score(text: str) -> float:
    """
    Detects technical/code/structured elements.
    """

    patterns = [
        r"```",
        r"\bdef\b",
        r"\bclass\b",
        r"\bimport\b",
        r"\btraceback\b",
        r"\berror\b",
        r"\bexception\b",
        r"\bjson\b",
        r"\bapi\b",
        r"\bsql\b",
        r"\bfunction\b",
        r"\{.*\}",
        r"\[.*\]"
    ]

    count = 0

    for pattern in patterns:
        if re.search(pattern, text.lower(), flags=re.DOTALL):
            count += 1

    return min(count / len(patterns), 1.0)

In [44]:
# Cell 43: Advanced Complexity Score

def advanced_complexity_score(query: str) -> Dict[str, Any]:
    """
    Replaces lexical-only complexity.
    Uses:
    - length score
    - local perplexity
    - structural signal
    - redundancy penalty
    """

    token_len = approx_tokens(query)
    ppl = calculate_perplexity(query)
    structural = structural_signal_score(query)
    red = redundancy_ratio(query)

    length_score = min(token_len / 1200, 1.0)
    ppl_score = min(math.log1p(ppl) / 8.0, 1.0)

    score = (
        0.30 * length_score
        + 0.40 * ppl_score
        + 0.40 * structural
        - 0.20 * red
    )

    score = max(0.05, min(score, 1.0))

    return {
        "advanced_complexity_score": round(score, 4),
        "perplexity": ppl,
        "ppl_score": round(ppl_score, 4),
        "structural_signal": round(structural, 4),
        "length_score": round(length_score, 4),
        "redundancy_ratio": round(red, 4),
    }

In [45]:
# Cell 44: Ridge Regression Budget Predictor

budget_training_data = []

def collect_budget_training_row(
    query: str,
    actual_output_tokens: int,
    quality_score: float
):
    """
    Collects historical optimization data.
    Later used to train ridge regression budget model.
    """

    adv = advanced_complexity_score(query)

    row = {
        "length_tokens": approx_tokens(query),
        "perplexity": adv["perplexity"],
        "structural_signal": adv["structural_signal"],
        "redundancy_ratio": adv["redundancy_ratio"],
        "quality_score": quality_score,
        "actual_output_tokens": actual_output_tokens
    }

    budget_training_data.append(row)


def bootstrap_budget_training_data():
    """
    Creates initial synthetic bootstrapping data from current test queries.
    Replace this with real logs once you have enough production runs.
    """

    global budget_training_data

    budget_training_data = []

    for use_case, query in test_queries.items():
        adv = advanced_complexity_score(query)

        heuristic_budget = allocate_output_budget(query)

        row = {
            "length_tokens": approx_tokens(query),
            "perplexity": adv["perplexity"],
            "structural_signal": adv["structural_signal"],
            "redundancy_ratio": adv["redundancy_ratio"],
            "quality_score": 0.70,
            "actual_output_tokens": heuristic_budget
        }

        budget_training_data.append(row)


bootstrap_budget_training_data()

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


In [46]:
# Cell 45: Train Ridge Budget Model

budget_scaler = StandardScaler()
budget_model = Ridge(alpha=1.0)


def train_budget_model():
    df_budget = pd.DataFrame(budget_training_data)

    X = df_budget[
        [
            "length_tokens",
            "perplexity",
            "structural_signal",
            "redundancy_ratio",
            "quality_score"
        ]
    ].values

    y = df_budget["actual_output_tokens"].values

    X_scaled = budget_scaler.fit_transform(X)
    budget_model.fit(X_scaled, y)

    return df_budget


budget_training_df = train_budget_model()
budget_training_df

,length_tokens,perplexity,structural_signal,redundancy_ratio,quality_score,actual_output_tokens
0,19,102.5879,0.0000,0.0,0.7,185
1,32,157.0285,0.1538,0.0,0.7,235
2,142,100.2358,0.0000,0.0,0.7,128
3,52,57.2030,0.0769,0.0,0.7,213
4,65,83.2932,0.1538,0.0,0.7,183
5,54,164.2775,0.0000,0.0,0.7,233


In [47]:
# Cell 46: Regression-Based Budget Allocation

def regression_budget_predictor(query: str, estimated_quality: float = 0.70) -> int:
    adv = advanced_complexity_score(query)

    features = np.array([
        [
            approx_tokens(query),
            adv["perplexity"],
            adv["structural_signal"],
            adv["redundancy_ratio"],
            estimated_quality
        ]
    ])

    features_scaled = budget_scaler.transform(features)

    predicted_budget = budget_model.predict(features_scaled)[0]

    # Safety clamp
    predicted_budget = int(max(120, min(predicted_budget, 800)))

    return predicted_budget

In [48]:
# Cell 47: Advanced Input Optimizer v2

def optimize_input_v2(query: str, use_memory: bool = False) -> Dict[str, Any]:
    """
    New optimizer:
    - Normalization
    - Embedding-based semantic pruning
    - Perplexity-aware complexity
    - Regression budget prediction
    """

    original = query

    q1 = normalize_text(query)
    q2 = embedding_semantic_prune(q1, q1, similarity_threshold=0.92)

    # Safety: never overcompress code/debug prompts
    code_markers = [
        "traceback",
        "error",
        "exception",
        "code:",
        "def ",
        "class ",
        "import "
    ]

    q_best = q2

    if any(m in original.lower() for m in code_markers):
        if approx_tokens(q_best) < approx_tokens(original) * 0.65:
            q_best = q1

    if use_memory:
        q_mem = memory_state.get_delta_prompt(q_best)

        if approx_tokens(q_mem) <= approx_tokens(q_best) * 1.10:
            q_best = q_mem

    adv = advanced_complexity_score(q_best)
    budget = regression_budget_predictor(q_best)

    original_tokens = approx_tokens(original)
    optimized_tokens = approx_tokens(q_best)

    return {
        "original_query": original,
        "optimized_query": q_best,
        "original_est_tokens": original_tokens,
        "optimized_est_tokens": optimized_tokens,
        "estimated_input_reduction_percent": round(
            ((original_tokens - optimized_tokens) / max(original_tokens, 1)) * 100,
            2
        ),
        "complexity_score": adv["advanced_complexity_score"],
        "perplexity": adv["perplexity"],
        "ppl_score": adv["ppl_score"],
        "structural_signal": adv["structural_signal"],
        "adaptive_output_budget": budget,
        "entropy": round(lexical_entropy(q_best), 4),
        "redundancy_ratio": adv["redundancy_ratio"],
        "information_density": round(information_density(q_best), 6),
    }

In [49]:
# Cell 48: Optimized Groq Call v2

def ask_groq_token_minimized_v2(
    query: str,
    model: str = MODEL,
    use_memory: bool = False,
) -> Dict[str, Any]:

    optimized = optimize_input_v2(query, use_memory=use_memory)
    max_tokens = optimized["adaptive_output_budget"]

    messages = [
        {
            "role": "system",
            "content": build_minimal_contract()
        },
        {
            "role": "user",
            "content": optimized["optimized_query"]
        }
    ]

    start = time.time()

    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0.0,
        top_p=0.68,
        max_completion_tokens=max_tokens,
        stop=STOP_SEQUENCES,
        seed=42,
    )

    answer = response.choices[0].message.content.strip()
    usage = response.usage

    result = {
        **optimized,
        "answer": answer,
        "input_tokens": usage.prompt_tokens,
        "output_tokens": usage.completion_tokens,
        "total_tokens": usage.total_tokens,
        "latency_seconds": round(time.time() - start, 3),
    }

    memory_state.update_state(query, answer)

    return result

In [50]:
# Cell 49: Compare Baseline vs v1 vs v2

def compare_v1_v2(query: str, use_memory: bool = False) -> Dict[str, Any]:
    baseline = ask_groq_baseline(query)
    v1 = ask_groq_token_minimized(query, use_memory=use_memory)
    v2 = ask_groq_token_minimized_v2(query, use_memory=use_memory)

    def reduction(base_tokens, optimized_tokens):
        return round(
            ((base_tokens - optimized_tokens) / max(base_tokens, 1)) * 100,
            2
        )

    return {
        "baseline": baseline,
        "v1": v1,
        "v2": v2,
        "v1_reduction_percent": reduction(
            baseline["total_tokens"],
            v1["total_tokens"]
        ),
        "v2_reduction_percent": reduction(
            baseline["total_tokens"],
            v2["total_tokens"]
        ),
    }


def print_compare_v1_v2(result: Dict[str, Any]):
    print("========== TOKEN COMPARISON ==========")
    print(f"Baseline Tokens : {result['baseline']['total_tokens']}")
    print(f"V1 Tokens       : {result['v1']['total_tokens']}")
    print(f"V2 Tokens       : {result['v2']['total_tokens']}")

    print("\n========== REDUCTION ==========")
    print(f"V1 Reduction %  : {result['v1_reduction_percent']}")
    print(f"V2 Reduction %  : {result['v2_reduction_percent']}")

    print("\n========== V2 METRICS ==========")
    print(f"Perplexity      : {result['v2']['perplexity']}")
    print(f"PPL Score       : {result['v2']['ppl_score']}")
    print(f"Structural      : {result['v2']['structural_signal']}")
    print(f"Budget          : {result['v2']['adaptive_output_budget']}")

    print("\n========== V2 ANSWER ==========\n")
    print(result["v2"]["answer"])

In [51]:
# Cell 50: Run One v2 Test

query = test_queries["debugging"]

v2_result = compare_v1_v2(query, use_memory=False)
print_compare_v1_v2(v2_result)

========== TOKEN COMPARISON ==========
Baseline Tokens : 256
V1 Tokens       : 135
V2 Tokens       : 124

========== REDUCTION ==========
V1 Reduction %  : 47.27
V2 Reduction %  : 51.56

========== V2 METRICS ==========
Perplexity      : 84.9635
PPL Score       : 0.5567
Structural      : 0.0769
Budget          : 201

========== V2 ANSWER ==========

```python
age = 25
message = "My age is " + str(age)
print(message)
```


In [52]:
# Cell 51: Batch v2 Evaluation

v2_rows = []

for use_case, query in test_queries.items():
    print(f"Running v2 evaluation: {use_case}")

    result = compare_v1_v2(query, use_memory=False)

    v2_quality = estimate_quality_score(
        result["v2"]["answer"],
        query
    )

    v2_cost = cost_usd(
        result["v2"]["input_tokens"],
        result["v2"]["output_tokens"]
    )

    baseline_cost = cost_usd(
        result["baseline"]["input_tokens"],
        result["baseline"]["output_tokens"]
    )

    cost_reduction = round(
        ((baseline_cost - v2_cost) / max(baseline_cost, 1e-12)) * 100,
        2
    )

    cf = compression_factor(
        result["v2"]["original_est_tokens"],
        result["v2"]["optimized_est_tokens"]
    )

    v2_tes = token_efficiency_score(
        information_density_value=result["v2"]["information_density"],
        quality_score=v2_quality,
        total_tokens=result["v2"]["total_tokens"]
    )

    v2_rows.append({
        "use_case": use_case,

        "baseline_total_tokens": result["baseline"]["total_tokens"],
        "v1_total_tokens": result["v1"]["total_tokens"],
        "v2_total_tokens": result["v2"]["total_tokens"],

        "v1_reduction_percent": result["v1_reduction_percent"],
        "v2_reduction_percent": result["v2_reduction_percent"],

        "v2_quality_score": v2_quality,
        "v2_tes": v2_tes,
        "v2_cost_usd": round(v2_cost, 8),
        "v2_cost_reduction_percent": cost_reduction,

        "v2_compression_factor": cf,
        "v2_perplexity": result["v2"]["perplexity"],
        "v2_structural_signal": result["v2"]["structural_signal"],
        "v2_complexity_score": result["v2"]["complexity_score"],
        "v2_budget": result["v2"]["adaptive_output_budget"],
    })

v2_df = pd.DataFrame(v2_rows)
v2_df

Running v2 evaluation: factual_qa
Running v2 evaluation: code_generation
Running v2 evaluation: summarization
Running v2 evaluation: debugging
Running v2 evaluation: json_extraction
Running v2 evaluation: creative


,use_case,baseline_total_tokens,v1_total_tokens,v2_total_tokens,v1_reduction_percent,v2_reduction_percent,v2_quality_score,v2_tes,v2_cost_usd,v2_cost_reduction_percent,v2_compression_factor,v2_perplexity,v2_structural_signal,v2_complexity_score,v2_budget
0,factual_qa,270,148,148,45.19,45.19,0.6183,0.000771,0.000103,49.62,1.0556,166.2022,0.0000,0.2605,229
1,code_generation,664,246,246,62.95,62.95,0.6000,0.000320,0.000178,65.35,1.0323,196.0030,0.1538,0.3334,247
2,summarization,209,192,241,8.13,-15.31,0.6765,0.000135,0.000161,-18.45,1.2137,166.3382,0.0000,0.2853,177
3,debugging,209,135,124,35.41,40.67,0.3404,0.000319,0.000078,47.08,1.4054,84.9635,0.0769,0.2627,201
4,json_extraction,207,169,189,18.36,8.70,0.6422,0.000311,0.000128,10.52,1.2500,110.7370,0.1538,0.3103,209
5,creative,584,182,184,68.84,68.49,0.7025,0.000349,0.000125,71.89,1.0189,200.4161,0.0000,0.2785,221


In [53]:
# Cell 52: Add GOES for v2

def add_goes_v2(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    df["tes_norm"] = df["v2_tes"].apply(
        lambda x: normalize_metric(
            x,
            df["v2_tes"].min(),
            df["v2_tes"].max()
        )
    )

    df["cost_reduction_norm"] = df["v2_cost_reduction_percent"].apply(
        lambda x: normalize_metric(
            x,
            df["v2_cost_reduction_percent"].min(),
            df["v2_cost_reduction_percent"].max()
        )
    )

    df["compression_norm"] = df["v2_compression_factor"].apply(
        lambda x: normalize_metric(
            x,
            df["v2_compression_factor"].min(),
            df["v2_compression_factor"].max()
        )
    )

    df["v2_goes"] = df.apply(
        lambda r: global_optimization_efficiency_score(
            tes_norm=r["tes_norm"],
            cost_reduction_norm=r["cost_reduction_norm"],
            quality_score=r["v2_quality_score"],
            compression_norm=r["compression_norm"]
        ),
        axis=1
    )

    return df


v2_df = add_goes_v2(v2_df)
v2_df.sort_values("v2_goes", ascending=False)

,use_case,baseline_total_tokens,v1_total_tokens,v2_total_tokens,v1_reduction_percent,v2_reduction_percent,v2_quality_score,v2_tes,v2_cost_usd,v2_cost_reduction_percent,v2_compression_factor,v2_perplexity,v2_structural_signal,v2_complexity_score,v2_budget,tes_norm,cost_reduction_norm,compression_norm,v2_goes
0,factual_qa,270,148,148,45.19,45.19,0.6183,0.000771,0.000103,49.62,1.0556,166.2022,0.0000,0.2605,229,1.000000,0.753487,0.094955,0.681023
3,debugging,209,135,124,35.41,40.67,0.3404,0.000319,0.000078,47.08,1.4054,84.9635,0.0769,0.2627,201,0.289295,0.725371,1.000000,0.550676
5,creative,584,182,184,68.84,68.49,0.7025,0.000349,0.000125,71.89,1.0189,200.4161,0.0000,0.2785,221,0.335457,1.000000,0.000000,0.507910
1,code_generation,664,246,246,62.95,62.95,0.6000,0.000320,0.000178,65.35,1.0323,196.0030,0.1538,0.3334,247,0.290837,0.927607,0.034670,0.460629
4,json_extraction,207,169,189,18.36,8.70,0.6422,0.000311,0.000128,10.52,1.2500,110.7370,0.1538,0.3103,209,0.275780,0.320677,0.597930,0.424718
2,summarization,209,192,241,8.13,-15.31,0.6765,0.000135,0.000161,-18.45,1.2137,166.3382,0.0000,0.2853,177,0.000000,0.000000,0.504010,0.236102


In [54]:
# Cell 53: v1 vs v2 Summary

v2_summary = {
    "avg_v1_token_reduction_percent": round(v2_df["v1_reduction_percent"].mean(), 2),
    "avg_v2_token_reduction_percent": round(v2_df["v2_reduction_percent"].mean(), 2),
    "avg_v2_cost_reduction_percent": round(v2_df["v2_cost_reduction_percent"].mean(), 2),
    "avg_v2_quality_score": round(v2_df["v2_quality_score"].mean(), 4),
    "avg_v2_goes": round(v2_df["v2_goes"].mean(), 4),
    "best_v2_use_case": v2_df.sort_values("v2_goes", ascending=False).iloc[0]["use_case"],
    "worst_v2_use_case": v2_df.sort_values("v2_goes", ascending=True).iloc[0]["use_case"],
}

v2_summary

{'avg_v1_token_reduction_percent': np.float64(39.81),
 'avg_v2_token_reduction_percent': np.float64(35.12),
 'avg_v2_cost_reduction_percent': np.float64(37.67),
 'avg_v2_quality_score': np.float64(0.5966),
 'avg_v2_goes': np.float64(0.4768),
 'best_v2_use_case': 'factual_qa',
 'worst_v2_use_case': 'summarization'}

In [55]:
# Cell 54: V2 Mathematical Fine-Tuning Search Space

V2_TUNING_GRID = {
    "similarity_threshold": [0.88, 0.90, 0.92, 0.94, 0.96],
    "top_p": [0.60, 0.64, 0.68, 0.72],
    "budget_min": [80, 100, 120],
    "budget_max": [420, 520, 620],
    "temperature": [0.0],
}

In [56]:
# Cell 55: Tunable Regression Budget Predictor

def regression_budget_predictor_tuned(
    query: str,
    estimated_quality: float = 0.70,
    budget_min: int = 100,
    budget_max: int = 520
) -> int:
    adv = advanced_complexity_score(query)

    features = np.array([
        [
            approx_tokens(query),
            adv["perplexity"],
            adv["structural_signal"],
            adv["redundancy_ratio"],
            estimated_quality
        ]
    ])

    features_scaled = budget_scaler.transform(features)
    predicted_budget = budget_model.predict(features_scaled)[0]

    predicted_budget = int(max(budget_min, min(predicted_budget, budget_max)))
    return predicted_budget

In [57]:
# Cell 56: Tunable V2 Input Optimizer

def optimize_input_v2_tuned(
    query: str,
    similarity_threshold: float = 0.92,
    budget_min: int = 100,
    budget_max: int = 520,
    use_memory: bool = False
) -> Dict[str, Any]:

    original = query

    q1 = normalize_text(query)
    q2 = embedding_semantic_prune(
        q1,
        q1,
        similarity_threshold=similarity_threshold
    )

    code_markers = [
        "traceback",
        "error",
        "exception",
        "code:",
        "def ",
        "class ",
        "import "
    ]

    q_best = q2

    # Safety constraint: preserve technical prompts
    if any(m in original.lower() for m in code_markers):
        if approx_tokens(q_best) < approx_tokens(original) * 0.70:
            q_best = q1

    if use_memory:
        q_mem = memory_state.get_delta_prompt(q_best)
        if approx_tokens(q_mem) <= approx_tokens(q_best) * 1.10:
            q_best = q_mem

    adv = advanced_complexity_score(q_best)

    budget = regression_budget_predictor_tuned(
        q_best,
        estimated_quality=0.70,
        budget_min=budget_min,
        budget_max=budget_max
    )

    original_tokens = approx_tokens(original)
    optimized_tokens = approx_tokens(q_best)

    return {
        "original_query": original,
        "optimized_query": q_best,
        "original_est_tokens": original_tokens,
        "optimized_est_tokens": optimized_tokens,
        "estimated_input_reduction_percent": round(
            ((original_tokens - optimized_tokens) / max(original_tokens, 1)) * 100,
            2
        ),
        "complexity_score": adv["advanced_complexity_score"],
        "perplexity": adv["perplexity"],
        "ppl_score": adv["ppl_score"],
        "structural_signal": adv["structural_signal"],
        "adaptive_output_budget": budget,
        "entropy": round(lexical_entropy(q_best), 4),
        "redundancy_ratio": adv["redundancy_ratio"],
        "information_density": round(information_density(q_best), 6),
    }

In [58]:
# Cell 57: Tunable V2 Groq Call

def ask_groq_token_minimized_v2_tuned(
    query: str,
    model: str = MODEL,
    similarity_threshold: float = 0.92,
    top_p: float = 0.68,
    budget_min: int = 100,
    budget_max: int = 520,
    temperature: float = 0.0,
    use_memory: bool = False,
) -> Dict[str, Any]:

    optimized = optimize_input_v2_tuned(
        query=query,
        similarity_threshold=similarity_threshold,
        budget_min=budget_min,
        budget_max=budget_max,
        use_memory=use_memory
    )

    messages = [
        {
            "role": "system",
            "content": build_minimal_contract()
        },
        {
            "role": "user",
            "content": optimized["optimized_query"]
        }
    ]

    start = time.time()

    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature,
        top_p=top_p,
        max_completion_tokens=optimized["adaptive_output_budget"],
        stop=STOP_SEQUENCES,
        seed=42,
    )

    answer = response.choices[0].message.content.strip()
    usage = response.usage

    result = {
        **optimized,
        "answer": answer,
        "input_tokens": usage.prompt_tokens,
        "output_tokens": usage.completion_tokens,
        "total_tokens": usage.total_tokens,
        "latency_seconds": round(time.time() - start, 3),
        "similarity_threshold": similarity_threshold,
        "top_p": top_p,
        "budget_min": budget_min,
        "budget_max": budget_max,
        "temperature": temperature,
    }

    memory_state.update_state(query, answer)
    return result

In [59]:
# Cell 58: Evaluate One Tuned V2 Configuration

def evaluate_v2_tuned_config(config: Dict[str, Any]) -> pd.DataFrame:
    rows = []

    for use_case, query in test_queries.items():
        baseline = ask_groq_baseline(query)
        v1 = ask_groq_token_minimized(query)
        v2t = ask_groq_token_minimized_v2_tuned(
            query=query,
            similarity_threshold=config["similarity_threshold"],
            top_p=config["top_p"],
            budget_min=config["budget_min"],
            budget_max=config["budget_max"],
            temperature=config["temperature"],
        )

        v2t_quality = estimate_quality_score(v2t["answer"], query)

        baseline_cost = cost_usd(
            baseline["input_tokens"],
            baseline["output_tokens"]
        )

        v1_cost = cost_usd(
            v1["input_tokens"],
            v1["output_tokens"]
        )

        v2t_cost = cost_usd(
            v2t["input_tokens"],
            v2t["output_tokens"]
        )

        v1_cost_reduction = round(
            ((baseline_cost - v1_cost) / max(baseline_cost, 1e-12)) * 100,
            2
        )

        v2t_cost_reduction = round(
            ((baseline_cost - v2t_cost) / max(baseline_cost, 1e-12)) * 100,
            2
        )

        v2t_token_reduction = round(
            ((baseline["total_tokens"] - v2t["total_tokens"]) / max(baseline["total_tokens"], 1)) * 100,
            2
        )

        v2t_cf = compression_factor(
            v2t["original_est_tokens"],
            v2t["optimized_est_tokens"]
        )

        v2t_tes = token_efficiency_score(
            information_density_value=v2t["information_density"],
            quality_score=v2t_quality,
            total_tokens=v2t["total_tokens"]
        )

        rows.append({
            "use_case": use_case,
            "baseline_total_tokens": baseline["total_tokens"],
            "v1_total_tokens": v1["total_tokens"],
            "v2t_total_tokens": v2t["total_tokens"],

            "v1_cost_reduction_percent": v1_cost_reduction,
            "v2t_cost_reduction_percent": v2t_cost_reduction,
            "v2t_token_reduction_percent": v2t_token_reduction,

            "v2t_quality_score": v2t_quality,
            "v2t_tes": v2t_tes,
            "v2t_compression_factor": v2t_cf,
            "v2t_complexity_score": v2t["complexity_score"],

            "similarity_threshold": config["similarity_threshold"],
            "top_p": config["top_p"],
            "budget_min": config["budget_min"],
            "budget_max": config["budget_max"],
            "temperature": config["temperature"],
        })

    df_config = pd.DataFrame(rows)

    df_config["tes_norm"] = df_config["v2t_tes"].apply(
        lambda x: normalize_metric(
            x,
            df_config["v2t_tes"].min(),
            df_config["v2t_tes"].max()
        )
    )

    df_config["cost_reduction_norm"] = df_config["v2t_cost_reduction_percent"].apply(
        lambda x: normalize_metric(
            x,
            df_config["v2t_cost_reduction_percent"].min(),
            df_config["v2t_cost_reduction_percent"].max()
        )
    )

    df_config["compression_norm"] = df_config["v2t_compression_factor"].apply(
        lambda x: normalize_metric(
            x,
            df_config["v2t_compression_factor"].min(),
            df_config["v2t_compression_factor"].max()
        )
    )

    df_config["v2t_goes"] = df_config.apply(
        lambda r: global_optimization_efficiency_score(
            tes_norm=r["tes_norm"],
            cost_reduction_norm=r["cost_reduction_norm"],
            quality_score=r["v2t_quality_score"],
            compression_norm=r["compression_norm"]
        ),
        axis=1
    )

    return df_config

In [60]:
# Cell 59: Mathematical Grid Search for V2

from itertools import product

def run_v2_tuning_grid(max_configs: int = None):
    configs = []

    for similarity_threshold, top_p, budget_min, budget_max, temperature in product(
        V2_TUNING_GRID["similarity_threshold"],
        V2_TUNING_GRID["top_p"],
        V2_TUNING_GRID["budget_min"],
        V2_TUNING_GRID["budget_max"],
        V2_TUNING_GRID["temperature"],
    ):
        if budget_min >= budget_max:
            continue

        configs.append({
            "similarity_threshold": similarity_threshold,
            "top_p": top_p,
            "budget_min": budget_min,
            "budget_max": budget_max,
            "temperature": temperature,
        })

    if max_configs:
        configs = configs[:max_configs]

    summary_rows = []
    detailed_results = {}

    for i, config in enumerate(configs, start=1):
        print(f"Running config {i}/{len(configs)}: {config}")

        df_config = evaluate_v2_tuned_config(config)

        config_key = json.dumps(config, sort_keys=True)
        detailed_results[config_key] = df_config

        avg_v2t_goes = df_config["v2t_goes"].mean()
        avg_v2t_cost_reduction = df_config["v2t_cost_reduction_percent"].mean()
        avg_v1_cost_reduction = df_config["v1_cost_reduction_percent"].mean()
        avg_v2t_quality = df_config["v2t_quality_score"].mean()
        avg_v2t_token_reduction = df_config["v2t_token_reduction_percent"].mean()

        # Constraint-aware objective
        # Penalize if tuned V2 is less cost-effective than V1
        cost_gap = avg_v2t_cost_reduction - avg_v1_cost_reduction
        cost_penalty = min(0, cost_gap) * 0.02

        objective = (
            0.45 * avg_v2t_goes
            + 0.25 * (avg_v2t_cost_reduction / 100)
            + 0.20 * avg_v2t_quality
            + 0.10 * (avg_v2t_token_reduction / 100)
            + cost_penalty
        )

        summary_rows.append({
            **config,
            "avg_v2t_goes": round(avg_v2t_goes, 4),
            "avg_v2t_cost_reduction_percent": round(avg_v2t_cost_reduction, 2),
            "avg_v1_cost_reduction_percent": round(avg_v1_cost_reduction, 2),
            "cost_gap_vs_v1": round(cost_gap, 2),
            "avg_v2t_quality_score": round(avg_v2t_quality, 4),
            "avg_v2t_token_reduction_percent": round(avg_v2t_token_reduction, 2),
            "objective_score": round(objective, 6),
        })

    tuning_summary_df = pd.DataFrame(summary_rows)
    tuning_summary_df = tuning_summary_df.sort_values(
        "objective_score",
        ascending=False
    ).reset_index(drop=True)

    return tuning_summary_df, detailed_results

In [61]:
# Cell 60: Run Small Tuning Test First

# Start with 10 configs first. Full grid will call Groq many times.
tuning_summary_df, detailed_results = run_v2_tuning_grid(max_configs=10)

tuning_summary_df

Running config 1/10: {'similarity_threshold': 0.88, 'top_p': 0.6, 'budget_min': 80, 'budget_max': 420, 'temperature': 0.0}


RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01jqzt8ryafhjtqye10djyrekx` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99852, Requested 541. Please try again in 5m39.552s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

In [ ]:
# Cell 61: Inspect Best Tuned V2 Configuration - Safe Version

best_row = tuning_summary_df.iloc[0]

best_config = {
    "similarity_threshold": float(best_row["similarity_threshold"]),
    "top_p": float(best_row["top_p"]),
    "budget_min": int(best_row["budget_min"]),
    "budget_max": int(best_row["budget_max"]),
    "temperature": float(best_row["temperature"]),
}

best_config

In [ ]:
# Cell 62: Detailed Results for Best Config - Safe Version

best_row = tuning_summary_df.iloc[0]

best_config = {
    "similarity_threshold": float(best_row["similarity_threshold"]),
    "top_p": float(best_row["top_p"]),
    "budget_min": int(best_row["budget_min"]),
    "budget_max": int(best_row["budget_max"]),
    "temperature": float(best_row["temperature"]),
}

best_key = json.dumps(best_config, sort_keys=True)

if best_key not in detailed_results:
    print("Exact key not found. Available keys:")
    for k in list(detailed_results.keys())[:5]:
        print(k)

    # fallback: pick first matching config approximately
    matched_key = None

    for k in detailed_results.keys():
        cfg = json.loads(k)

        if (
            abs(float(cfg["similarity_threshold"]) - best_config["similarity_threshold"]) < 1e-9
            and abs(float(cfg["top_p"]) - best_config["top_p"]) < 1e-9
            and int(cfg["budget_min"]) == best_config["budget_min"]
            and int(cfg["budget_max"]) == best_config["budget_max"]
            and abs(float(cfg["temperature"]) - best_config["temperature"]) < 1e-9
        ):
            matched_key = k
            break

    if matched_key is None:
        raise KeyError("No matching config found in detailed_results.")

    best_key = matched_key

best_v2t_df = detailed_results[best_key]

best_v2t_df.sort_values("v2t_goes", ascending=False)

In [ ]:
# Cell 63: Final V1 vs V2 vs Tuned V2 Summary

final_tuning_summary = {
    "best_similarity_threshold": best_config["similarity_threshold"],
    "best_top_p": best_config["top_p"],
    "best_budget_min": best_config["budget_min"],
    "best_budget_max": best_config["budget_max"],

    "avg_v1_cost_reduction_percent": round(best_v2t_df["v1_cost_reduction_percent"].mean(), 2),
    "avg_tuned_v2_cost_reduction_percent": round(best_v2t_df["v2t_cost_reduction_percent"].mean(), 2),
    "cost_gap_vs_v1": round(
        best_v2t_df["v2t_cost_reduction_percent"].mean()
        - best_v2t_df["v1_cost_reduction_percent"].mean(),
        2
    ),

    "avg_tuned_v2_token_reduction_percent": round(best_v2t_df["v2t_token_reduction_percent"].mean(), 2),
    "avg_tuned_v2_quality_score": round(best_v2t_df["v2t_quality_score"].mean(), 4),
    "avg_tuned_v2_goes": round(best_v2t_df["v2t_goes"].mean(), 4),
}

final_tuning_summary

In [ ]:
# Cell 64: Final Interpretation

print("="*80)
print("FINAL V1 vs TUNED V2 ANALYSIS")
print("="*80)

print(f"\nBest Similarity Threshold : {final_tuning_summary['best_similarity_threshold']}")
print(f"Best Top-p               : {final_tuning_summary['best_top_p']}")
print(f"Best Budget Min          : {final_tuning_summary['best_budget_min']}")
print(f"Best Budget Max          : {final_tuning_summary['best_budget_max']}")

print("\n" + "="*80)
print("COST ANALYSIS")
print("="*80)

print(f"Average V1 Cost Reduction      : {final_tuning_summary['avg_v1_cost_reduction_percent']:.2f}%")
print(f"Average Tuned V2 Cost Reduction: {final_tuning_summary['avg_tuned_v2_cost_reduction_percent']:.2f}%")

cost_gap = final_tuning_summary["cost_gap_vs_v1"]

if cost_gap > 0:
    print(f"\n✅ Tuned V2 is MORE cost-efficient than V1 by {cost_gap:.2f}%")

elif cost_gap < 0:
    print(f"\n⚠️ Tuned V2 is LESS cost-efficient than V1 by {abs(cost_gap):.2f}%")

else:
    print("\n➖ Tuned V2 and V1 have identical cost efficiency")

print("\n" + "="*80)
print("QUALITY ANALYSIS")
print("="*80)

print(f"Average Tuned V2 Quality Score : {final_tuning_summary['avg_tuned_v2_quality_score']:.4f}")

if final_tuning_summary["avg_tuned_v2_quality_score"] >= 0.65:
    print("✅ Quality is strong")

elif final_tuning_summary["avg_tuned_v2_quality_score"] >= 0.50:
    print("⚠️ Quality is acceptable but can improve")

else:
    print("❌ Quality degradation detected")

print("\n" + "="*80)
print("TOKEN ANALYSIS")
print("="*80)

print(f"Average Tuned V2 Token Reduction : {final_tuning_summary['avg_tuned_v2_token_reduction_percent']:.2f}%")

if final_tuning_summary["avg_tuned_v2_token_reduction_percent"] >= 50:
    print("✅ Excellent token optimization")

elif final_tuning_summary["avg_tuned_v2_token_reduction_percent"] >= 30:
    print("⚠️ Moderate token optimization")

else:
    print("❌ Weak token optimization")

print("\n" + "="*80)
print("GOES ANALYSIS")
print("="*80)

print(f"Average Tuned V2 GOES : {final_tuning_summary['avg_tuned_v2_goes']:.4f}")

if final_tuning_summary["avg_tuned_v2_goes"] >= 0.60:
    print("✅ Strong optimization framework")

elif final_tuning_summary["avg_tuned_v2_goes"] >= 0.45:
    print("⚠️ Reasonable optimization framework")

else:
    print("❌ Optimization framework still weak")

print("\n" + "="*80)
print("FINAL VERDICT")
print("="*80)

if cost_gap >= 0:
    print("""
Recommendation:
Use Tuned V2.

Reason:
- Maintains quality
- Matches or exceeds V1 cost savings
- Uses semantic pruning
- Uses perplexity-aware complexity estimation
- Uses regression-based budgeting
""")
else:
    print(f"""
Recommendation:
Use V1 in production today.

Reason:
- V1 saves {abs(cost_gap):.2f}% more cost than Tuned V2
- Tuned V2 architecture is theoretically stronger
- More tuning and real training data are required

Current state:
V2 is safer but not yet economically superior.
""")

In [ ]:
# Cell 65: Semantic Quality + Preservation Scoring

def embedding_cosine_similarity(text_a: str, text_b: str) -> float:
    emb = embedding_model.encode([text_a, text_b], convert_to_numpy=True)

    a = emb[0]
    b = emb[1]

    denom = np.linalg.norm(a) * np.linalg.norm(b)

    if denom == 0:
        return 0.0

    return round(float(np.dot(a, b) / denom), 4)


def semantic_quality_score(answer: str, query: str) -> float:
    """
    Better quality proxy than pure term overlap.
    Combines:
    - semantic similarity
    - lexical coverage
    - anti-repetition
    """

    semantic_sim = embedding_cosine_similarity(query, answer)

    query_terms = set(re.findall(r"\w+", query.lower()))
    answer_terms = set(re.findall(r"\w+", answer.lower()))

    lexical_coverage = len(query_terms & answer_terms) / max(len(query_terms), 1)
    repetition_penalty = redundancy_ratio(answer)

    quality = (
        0.55 * semantic_sim
        + 0.30 * lexical_coverage
        + 0.15 * (1 - repetition_penalty)
    )

    return round(max(0.0, min(quality, 1.0)), 4)


def preservation_score(original_query: str, optimized_query: str) -> float:
    """
    Measures whether optimized input preserved original meaning.
    """

    return embedding_cosine_similarity(original_query, optimized_query)

In [ ]:
# Cell 66: Preservation-Gated V2 Input Optimizer

def optimize_input_v2_1(
    query: str,
    similarity_threshold: float = 0.88,
    preservation_threshold: float = 0.90,
    budget_min: int = 100,
    budget_max: int = 520,
    use_memory: bool = False,
) -> Dict[str, Any]:

    original = query

    q1 = normalize_text(query)

    q2 = embedding_semantic_prune(
        q1,
        q1,
        similarity_threshold=similarity_threshold
    )

    preserve = preservation_score(q1, q2)

    # Safety: if semantic preservation is weak, fallback
    if preserve < preservation_threshold:
        q_best = q1
        compression_accepted = False
    else:
        q_best = q2
        compression_accepted = True

    code_markers = [
        "traceback",
        "error",
        "exception",
        "code:",
        "def ",
        "class ",
        "import "
    ]

    # Stronger safety for technical prompts
    if any(m in original.lower() for m in code_markers):
        if approx_tokens(q_best) < approx_tokens(original) * 0.70:
            q_best = q1
            compression_accepted = False

    if use_memory:
        q_mem = memory_state.get_delta_prompt(q_best)
        if approx_tokens(q_mem) <= approx_tokens(q_best) * 1.10:
            q_best = q_mem

    adv = advanced_complexity_score(q_best)

    budget = regression_budget_predictor_tuned(
        query=q_best,
        estimated_quality=0.70,
        budget_min=budget_min,
        budget_max=budget_max
    )

    original_tokens = approx_tokens(original)
    optimized_tokens = approx_tokens(q_best)

    return {
        "original_query": original,
        "optimized_query": q_best,
        "original_est_tokens": original_tokens,
        "optimized_est_tokens": optimized_tokens,
        "estimated_input_reduction_percent": round(
            ((original_tokens - optimized_tokens) / max(original_tokens, 1)) * 100,
            2
        ),
        "preservation_score": preserve,
        "compression_accepted": compression_accepted,
        "complexity_score": adv["advanced_complexity_score"],
        "perplexity": adv["perplexity"],
        "ppl_score": adv["ppl_score"],
        "structural_signal": adv["structural_signal"],
        "adaptive_output_budget": budget,
        "entropy": round(lexical_entropy(q_best), 4),
        "redundancy_ratio": adv["redundancy_ratio"],
        "information_density": round(information_density(q_best), 6),
    }

In [ ]:
# Cell 67: V2.1 Groq Call

def ask_groq_token_minimized_v2_1(
    query: str,
    model: str = MODEL,
    similarity_threshold: float = 0.88,
    preservation_threshold: float = 0.90,
    budget_min: int = 100,
    budget_max: int = 520,
    top_p: float = 0.60,
    temperature: float = 0.0,
    use_memory: bool = False,
) -> Dict[str, Any]:

    optimized = optimize_input_v2_1(
        query=query,
        similarity_threshold=similarity_threshold,
        preservation_threshold=preservation_threshold,
        budget_min=budget_min,
        budget_max=budget_max,
        use_memory=use_memory,
    )

    messages = [
        {
            "role": "system",
            "content": build_minimal_contract()
        },
        {
            "role": "user",
            "content": optimized["optimized_query"]
        }
    ]

    start = time.time()

    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature,
        top_p=top_p,
        max_completion_tokens=optimized["adaptive_output_budget"],
        stop=STOP_SEQUENCES,
        seed=42,
    )

    answer = response.choices[0].message.content.strip()
    usage = response.usage

    result = {
        **optimized,
        "answer": answer,
        "input_tokens": usage.prompt_tokens,
        "output_tokens": usage.completion_tokens,
        "total_tokens": usage.total_tokens,
        "latency_seconds": round(time.time() - start, 3),
        "similarity_threshold": similarity_threshold,
        "preservation_threshold": preservation_threshold,
        "top_p": top_p,
        "budget_min": budget_min,
        "budget_max": budget_max,
        "temperature": temperature,
    }

    memory_state.update_state(query, answer)

    return result

In [ ]:
# Cell 68: Compare V1 vs V2.1

def compare_v1_v2_1(query: str) -> Dict[str, Any]:
    baseline = ask_groq_baseline(query)
    v1 = ask_groq_token_minimized(query)
    v21 = ask_groq_token_minimized_v2_1(query)

    baseline_cost = cost_usd(
        baseline["input_tokens"],
        baseline["output_tokens"]
    )

    v1_cost = cost_usd(
        v1["input_tokens"],
        v1["output_tokens"]
    )

    v21_cost = cost_usd(
        v21["input_tokens"],
        v21["output_tokens"]
    )

    v1_quality = semantic_quality_score(v1["answer"], query)
    v21_quality = semantic_quality_score(v21["answer"], query)

    return {
        "baseline": baseline,
        "v1": v1,
        "v21": v21,

        "v1_cost_reduction_percent": round(
            ((baseline_cost - v1_cost) / baseline_cost) * 100,
            2
        ),
        "v21_cost_reduction_percent": round(
            ((baseline_cost - v21_cost) / baseline_cost) * 100,
            2
        ),

        "v1_quality": v1_quality,
        "v21_quality": v21_quality,

        "v1_token_reduction_percent": round(
            ((baseline["total_tokens"] - v1["total_tokens"]) / baseline["total_tokens"]) * 100,
            2
        ),
        "v21_token_reduction_percent": round(
            ((baseline["total_tokens"] - v21["total_tokens"]) / baseline["total_tokens"]) * 100,
            2
        ),
    }

In [ ]:
# Cell 69: Batch Evaluate V2.1

v21_rows = []

for use_case, query in test_queries.items():
    print(f"Running V2.1 evaluation: {use_case}")

    result = compare_v1_v2_1(query)

    v21 = result["v21"]

    cf = compression_factor(
        v21["original_est_tokens"],
        v21["optimized_est_tokens"]
    )

    tes = token_efficiency_score(
        information_density_value=v21["information_density"],
        quality_score=result["v21_quality"],
        total_tokens=v21["total_tokens"]
    )

    v21_rows.append({
        "use_case": use_case,

        "v1_cost_reduction_percent": result["v1_cost_reduction_percent"],
        "v21_cost_reduction_percent": result["v21_cost_reduction_percent"],
        "cost_gap_vs_v1": round(
            result["v21_cost_reduction_percent"] - result["v1_cost_reduction_percent"],
            2
        ),

        "v1_token_reduction_percent": result["v1_token_reduction_percent"],
        "v21_token_reduction_percent": result["v21_token_reduction_percent"],

        "v1_quality": result["v1_quality"],
        "v21_quality": result["v21_quality"],
        "quality_gap_vs_v1": round(
            result["v21_quality"] - result["v1_quality"],
            4
        ),

        "v21_preservation_score": v21["preservation_score"],
        "compression_accepted": v21["compression_accepted"],

        "v21_tes": tes,
        "v21_compression_factor": cf,
        "v21_complexity_score": v21["complexity_score"],
        "v21_total_tokens": v21["total_tokens"],
    })

v21_df = pd.DataFrame(v21_rows)
v21_df

In [ ]:
# Cell 70: Add GOES for V2.1

def add_goes_v21(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    df["tes_norm"] = df["v21_tes"].apply(
        lambda x: normalize_metric(x, df["v21_tes"].min(), df["v21_tes"].max())
    )

    df["cost_reduction_norm"] = df["v21_cost_reduction_percent"].apply(
        lambda x: normalize_metric(
            x,
            df["v21_cost_reduction_percent"].min(),
            df["v21_cost_reduction_percent"].max()
        )
    )

    df["compression_norm"] = df["v21_compression_factor"].apply(
        lambda x: normalize_metric(
            x,
            df["v21_compression_factor"].min(),
            df["v21_compression_factor"].max()
        )
    )

    df["v21_goes"] = df.apply(
        lambda r: global_optimization_efficiency_score(
            tes_norm=r["tes_norm"],
            cost_reduction_norm=r["cost_reduction_norm"],
            quality_score=r["v21_quality"],
            compression_norm=r["compression_norm"]
        ),
        axis=1
    )

    return df


v21_df = add_goes_v21(v21_df)
v21_df.sort_values("v21_goes", ascending=False)

In [ ]:
# Cell 71: Final V2.1 Interpretation

v21_summary = {
    "avg_v1_cost_reduction_percent": round(v21_df["v1_cost_reduction_percent"].mean(), 2),
    "avg_v21_cost_reduction_percent": round(v21_df["v21_cost_reduction_percent"].mean(), 2),
    "avg_cost_gap_vs_v1": round(v21_df["cost_gap_vs_v1"].mean(), 2),

    "avg_v1_quality": round(v21_df["v1_quality"].mean(), 4),
    "avg_v21_quality": round(v21_df["v21_quality"].mean(), 4),
    "avg_quality_gap_vs_v1": round(v21_df["quality_gap_vs_v1"].mean(), 4),

    "avg_v21_token_reduction_percent": round(v21_df["v21_token_reduction_percent"].mean(), 2),
    "avg_v21_preservation_score": round(v21_df["v21_preservation_score"].mean(), 4),
    "avg_v21_goes": round(v21_df["v21_goes"].mean(), 4),
}

v21_summary

In [ ]:
# Cell 72: Human Interpretation for V2.1

print("="*80)
print("FINAL V1 vs V2.1 PRODUCTION READINESS ANALYSIS")
print("="*80)

print(f"Average V1 Cost Reduction    : {v21_summary['avg_v1_cost_reduction_percent']}%")
print(f"Average V2.1 Cost Reduction  : {v21_summary['avg_v21_cost_reduction_percent']}%")
print(f"Cost Gap vs V1               : {v21_summary['avg_cost_gap_vs_v1']}%")

print("\nQuality:")
print(f"Average V1 Quality           : {v21_summary['avg_v1_quality']}")
print(f"Average V2.1 Quality         : {v21_summary['avg_v21_quality']}")
print(f"Quality Gap vs V1            : {v21_summary['avg_quality_gap_vs_v1']}")

print("\nSemantic Safety:")
print(f"Average Preservation Score   : {v21_summary['avg_v21_preservation_score']}")

print("\nOptimization:")
print(f"Average V2.1 Token Reduction : {v21_summary['avg_v21_token_reduction_percent']}%")
print(f"Average V2.1 GOES            : {v21_summary['avg_v21_goes']}")

print("\nVerdict:")

if v21_summary["avg_cost_gap_vs_v1"] >= 0 and v21_summary["avg_quality_gap_vs_v1"] >= 0:
    print("✅ V2.1 is production-ready and better than V1.")
elif v21_summary["avg_quality_gap_vs_v1"] > 0 and v21_summary["avg_cost_gap_vs_v1"] > -5:
    print("⚠️ V2.1 is production-viable when semantic safety matters.")
else:
    print("❌ V1 remains better for production cost efficiency today.")

In [ ]:
# Cell 73: V3 Hybrid Routing Features

def query_routing_features(query: str) -> Dict[str, Any]:
    """
    Extracts mathematical routing features for V3.
    No LLM call.
    """

    q = query.lower()

    adv = advanced_complexity_score(query)

    token_len = approx_tokens(query)
    struct = adv["structural_signal"]
    ppl_score = adv["ppl_score"]
    complexity = adv["advanced_complexity_score"]
    red = redundancy_ratio(query)

    code_debug_signal = int(any(x in q for x in [
        "error",
        "exception",
        "traceback",
        "bug",
        "fix",
        "code:",
        "def ",
        "class ",
        "import "
    ]))

    extraction_signal = int(any(x in q for x in [
        "json",
        "extract",
        "schema",
        "fields",
        "parse"
    ]))

    summarization_signal = int(any(x in q for x in [
        "summarize",
        "summary",
        "compress"
    ]))

    creative_signal = int(any(x in q for x in [
        "linkedin",
        "creative",
        "post",
        "story",
        "caption"
    ]))

    return {
        "token_len": token_len,
        "complexity_score": complexity,
        "ppl_score": ppl_score,
        "structural_signal": struct,
        "redundancy_ratio": red,
        "code_debug_signal": code_debug_signal,
        "extraction_signal": extraction_signal,
        "summarization_signal": summarization_signal,
        "creative_signal": creative_signal,
    }

In [ ]:
# Cell 74: V3 Hybrid Router

def choose_optimizer_v3(query: str) -> Dict[str, Any]:
    """
    Hybrid decision policy.

    Logic:
    - Use V2.1 when semantic safety matters.
    - Use V1 when query is simple or cost-sensitive.
    """

    f = query_routing_features(query)

    semantic_risk_score = (
        0.35 * f["complexity_score"]
        + 0.25 * f["structural_signal"]
        + 0.20 * f["ppl_score"]
        + 0.20 * f["code_debug_signal"]
    )

    compression_opportunity_score = (
        0.45 * f["redundancy_ratio"]
        + 0.25 * min(f["token_len"] / 600, 1.0)
        + 0.15 * f["summarization_signal"]
        + 0.15 * f["creative_signal"]
    )

    # Default: V1 for cost
    selected = "v1"
    reason = "Low semantic risk; aggressive cost optimization preferred."

    # V2.1 for risky technical prompts
    if semantic_risk_score >= 0.42:
        selected = "v21"
        reason = "High semantic risk; preservation-gated optimizer preferred."

    # Summarization often suffers from over-compression
    if f["summarization_signal"] == 1:
        selected = "v21"
        reason = "Summarization requires semantic preservation; avoiding aggressive pruning."

    # Simple factual / creative queries generally benefit from V1
    if (
        semantic_risk_score < 0.35
        and compression_opportunity_score >= 0.10
        and f["code_debug_signal"] == 0
    ):
        selected = "v1"
        reason = "Good compression opportunity with low semantic risk."

    return {
        **f,
        "semantic_risk_score": round(semantic_risk_score, 4),
        "compression_opportunity_score": round(compression_opportunity_score, 4),
        "selected_optimizer": selected,
        "routing_reason": reason,
    }

In [ ]:
# Cell 75: V3 Hybrid Optimized Groq Call

def ask_groq_token_minimized_v3(
    query: str,
    model: str = MODEL,
) -> Dict[str, Any]:

    route = choose_optimizer_v3(query)

    if route["selected_optimizer"] == "v21":
        result = ask_groq_token_minimized_v2_1(
            query=query,
            model=model
        )
    else:
        result = ask_groq_token_minimized(
            query=query,
            model=model
        )

    result["v3_selected_optimizer"] = route["selected_optimizer"]
    result["v3_routing_reason"] = route["routing_reason"]
    result["semantic_risk_score"] = route["semantic_risk_score"]
    result["compression_opportunity_score"] = route["compression_opportunity_score"]

    return result

In [ ]:
# Cell 76: Compare V1 vs V2.1 vs V3

def compare_v1_v21_v3(query: str) -> Dict[str, Any]:
    baseline = ask_groq_baseline(query)
    v1 = ask_groq_token_minimized(query)
    v21 = ask_groq_token_minimized_v2_1(query)
    v3 = ask_groq_token_minimized_v3(query)

    baseline_cost = cost_usd(
        baseline["input_tokens"],
        baseline["output_tokens"]
    )

    v1_cost = cost_usd(
        v1["input_tokens"],
        v1["output_tokens"]
    )

    v21_cost = cost_usd(
        v21["input_tokens"],
        v21["output_tokens"]
    )

    v3_cost = cost_usd(
        v3["input_tokens"],
        v3["output_tokens"]
    )

    v1_quality = semantic_quality_score(v1["answer"], query)
    v21_quality = semantic_quality_score(v21["answer"], query)
    v3_quality = semantic_quality_score(v3["answer"], query)

    return {
        "baseline": baseline,
        "v1": v1,
        "v21": v21,
        "v3": v3,

        "baseline_cost": baseline_cost,
        "v1_cost": v1_cost,
        "v21_cost": v21_cost,
        "v3_cost": v3_cost,

        "v1_cost_reduction_percent": round(
            ((baseline_cost - v1_cost) / baseline_cost) * 100,
            2
        ),
        "v21_cost_reduction_percent": round(
            ((baseline_cost - v21_cost) / baseline_cost) * 100,
            2
        ),
        "v3_cost_reduction_percent": round(
            ((baseline_cost - v3_cost) / baseline_cost) * 100,
            2
        ),

        "v1_quality": v1_quality,
        "v21_quality": v21_quality,
        "v3_quality": v3_quality,

        "v1_token_reduction_percent": round(
            ((baseline["total_tokens"] - v1["total_tokens"]) / baseline["total_tokens"]) * 100,
            2
        ),
        "v21_token_reduction_percent": round(
            ((baseline["total_tokens"] - v21["total_tokens"]) / baseline["total_tokens"]) * 100,
            2
        ),
        "v3_token_reduction_percent": round(
            ((baseline["total_tokens"] - v3["total_tokens"]) / baseline["total_tokens"]) * 100,
            2
        ),
    }

In [ ]:
# Cell 77: Batch Evaluate V3 Hybrid

v3_rows = []

for use_case, query in test_queries.items():
    print(f"Running V3 hybrid evaluation: {use_case}")

    result = compare_v1_v21_v3(query)

    v3 = result["v3"]

    cf = compression_factor(
        v3["original_est_tokens"],
        v3["optimized_est_tokens"]
    )

    tes = token_efficiency_score(
        information_density_value=v3["information_density"],
        quality_score=result["v3_quality"],
        total_tokens=v3["total_tokens"]
    )

    v3_rows.append({
        "use_case": use_case,

        "selected_optimizer": v3["v3_selected_optimizer"],
        "routing_reason": v3["v3_routing_reason"],

        "semantic_risk_score": v3["semantic_risk_score"],
        "compression_opportunity_score": v3["compression_opportunity_score"],

        "v1_cost_reduction_percent": result["v1_cost_reduction_percent"],
        "v21_cost_reduction_percent": result["v21_cost_reduction_percent"],
        "v3_cost_reduction_percent": result["v3_cost_reduction_percent"],

        "v3_cost_gap_vs_v1": round(
            result["v3_cost_reduction_percent"] - result["v1_cost_reduction_percent"],
            2
        ),
        "v3_cost_gap_vs_v21": round(
            result["v3_cost_reduction_percent"] - result["v21_cost_reduction_percent"],
            2
        ),

        "v1_quality": result["v1_quality"],
        "v21_quality": result["v21_quality"],
        "v3_quality": result["v3_quality"],

        "v3_quality_gap_vs_v1": round(
            result["v3_quality"] - result["v1_quality"],
            4
        ),
        "v3_quality_gap_vs_v21": round(
            result["v3_quality"] - result["v21_quality"],
            4
        ),

        "v1_token_reduction_percent": result["v1_token_reduction_percent"],
        "v21_token_reduction_percent": result["v21_token_reduction_percent"],
        "v3_token_reduction_percent": result["v3_token_reduction_percent"],

        "v3_tes": tes,
        "v3_compression_factor": cf,
        "v3_complexity_score": v3["complexity_score"],
        "v3_total_tokens": v3["total_tokens"],
    })

v3_df = pd.DataFrame(v3_rows)
v3_df

In [ ]:
# Cell 78: Add GOES for V3

def add_goes_v3(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    df["tes_norm"] = df["v3_tes"].apply(
        lambda x: normalize_metric(
            x,
            df["v3_tes"].min(),
            df["v3_tes"].max()
        )
    )

    df["cost_reduction_norm"] = df["v3_cost_reduction_percent"].apply(
        lambda x: normalize_metric(
            x,
            df["v3_cost_reduction_percent"].min(),
            df["v3_cost_reduction_percent"].max()
        )
    )

    df["compression_norm"] = df["v3_compression_factor"].apply(
        lambda x: normalize_metric(
            x,
            df["v3_compression_factor"].min(),
            df["v3_compression_factor"].max()
        )
    )

    df["v3_goes"] = df.apply(
        lambda r: global_optimization_efficiency_score(
            tes_norm=r["tes_norm"],
            cost_reduction_norm=r["cost_reduction_norm"],
            quality_score=r["v3_quality"],
            compression_norm=r["compression_norm"]
        ),
        axis=1
    )

    return df


v3_df = add_goes_v3(v3_df)
v3_df.sort_values("v3_goes", ascending=False)

In [ ]:
# Cell 79: V3 Summary

v3_summary = {
    "avg_v1_cost_reduction_percent": round(v3_df["v1_cost_reduction_percent"].mean(), 2),
    "avg_v21_cost_reduction_percent": round(v3_df["v21_cost_reduction_percent"].mean(), 2),
    "avg_v3_cost_reduction_percent": round(v3_df["v3_cost_reduction_percent"].mean(), 2),

    "avg_v1_quality": round(v3_df["v1_quality"].mean(), 4),
    "avg_v21_quality": round(v3_df["v21_quality"].mean(), 4),
    "avg_v3_quality": round(v3_df["v3_quality"].mean(), 4),

    "avg_v3_token_reduction_percent": round(v3_df["v3_token_reduction_percent"].mean(), 2),
    "avg_v3_goes": round(v3_df["v3_goes"].mean(), 4),

    "v3_selected_v1_count": int((v3_df["selected_optimizer"] == "v1").sum()),
    "v3_selected_v21_count": int((v3_df["selected_optimizer"] == "v21").sum()),
}

v3_summary

In [ ]:
# Cell 80: Final V3 Interpretation

print("="*80)
print("FINAL V3 HYBRID OPTIMIZER ANALYSIS")
print("="*80)

print("\nCost Reduction:")
print(f"Average V1 Cost Reduction    : {v3_summary['avg_v1_cost_reduction_percent']}%")
print(f"Average V2.1 Cost Reduction  : {v3_summary['avg_v21_cost_reduction_percent']}%")
print(f"Average V3 Cost Reduction    : {v3_summary['avg_v3_cost_reduction_percent']}%")

print("\nQuality:")
print(f"Average V1 Quality           : {v3_summary['avg_v1_quality']}")
print(f"Average V2.1 Quality         : {v3_summary['avg_v21_quality']}")
print(f"Average V3 Quality           : {v3_summary['avg_v3_quality']}")

print("\nToken Reduction:")
print(f"Average V3 Token Reduction   : {v3_summary['avg_v3_token_reduction_percent']}%")

print("\nGOES:")
print(f"Average V3 GOES              : {v3_summary['avg_v3_goes']}")

print("\nRouting Distribution:")
print(f"V3 selected V1               : {v3_summary['v3_selected_v1_count']} cases")
print(f"V3 selected V2.1             : {v3_summary['v3_selected_v21_count']} cases")

print("\nVerdict:")

if (
    v3_summary["avg_v3_cost_reduction_percent"] >= v3_summary["avg_v1_cost_reduction_percent"]
    and v3_summary["avg_v3_quality"] >= v3_summary["avg_v1_quality"]
):
    print("✅ V3 is better than V1 and production-ready.")

elif (
    v3_summary["avg_v3_quality"] >= v3_summary["avg_v21_quality"]
    and v3_summary["avg_v3_cost_reduction_percent"] >= v3_summary["avg_v21_cost_reduction_percent"]
):
    print("✅ V3 improves over V2.1 and is production-viable.")

elif (
    v3_summary["avg_v3_cost_reduction_percent"] >= v3_summary["avg_v1_cost_reduction_percent"] - 3
    and v3_summary["avg_v3_quality"] > v3_summary["avg_v1_quality"]
):
    print("⚠️ V3 is a balanced production candidate.")

else:
    print("❌ V3 needs router threshold tuning.")

In [ ]:
# Cell 81: Inspect Routing Decisions

v3_df[
    [
        "use_case",
        "selected_optimizer",
        "semantic_risk_score",
        "compression_opportunity_score",
        "v1_cost_reduction_percent",
        "v21_cost_reduction_percent",
        "v3_cost_reduction_percent",
        "v1_quality",
        "v21_quality",
        "v3_quality",
        "v3_goes",
        "routing_reason"
    ]
].sort_values("v3_goes", ascending=False)

In [ ]:
# Cell 82: Oracle Router Analysis

def oracle_select(row, quality_margin=0.01, cost_margin=3.0):
    """
    Chooses the best optimizer per row using actual observed V1 and V2.1 results.

    Rule:
    - Prefer V2.1 if quality gain is meaningful and cost loss is acceptable.
    - Otherwise use V1.
    """

    quality_gain = row["v21_quality"] - row["v1_quality"]
    cost_gap = row["v21_cost_reduction_percent"] - row["v1_cost_reduction_percent"]

    if quality_gain >= quality_margin and cost_gap >= -cost_margin:
        return "v21"

    return "v1"


v3_df["oracle_optimizer"] = v3_df.apply(oracle_select, axis=1)

v3_df[
    [
        "use_case",
        "selected_optimizer",
        "oracle_optimizer",
        "semantic_risk_score",
        "compression_opportunity_score",
        "v1_cost_reduction_percent",
        "v21_cost_reduction_percent",
        "v1_quality",
        "v21_quality"
    ]
]

In [ ]:
# Cell 83: Learn Better Router Thresholds from Existing Results

candidate_risk_thresholds = [0.18, 0.22, 0.26, 0.30, 0.34, 0.38, 0.42]
candidate_compression_thresholds = [0.00, 0.03, 0.06, 0.10, 0.15, 0.20]

router_tuning_rows = []

for risk_threshold in candidate_risk_thresholds:
    for compression_threshold in candidate_compression_thresholds:

        predicted = []

        for _, row in v3_df.iterrows():
            if row["semantic_risk_score"] >= risk_threshold:
                pred = "v21"
            elif row["compression_opportunity_score"] >= compression_threshold:
                pred = "v1"
            else:
                pred = "v1"

            predicted.append(pred)

        temp = v3_df.copy()
        temp["predicted_optimizer"] = predicted

        accuracy = (temp["predicted_optimizer"] == temp["oracle_optimizer"]).mean()

        router_tuning_rows.append({
            "risk_threshold": risk_threshold,
            "compression_threshold": compression_threshold,
            "router_accuracy": round(accuracy, 4),
            "v21_selected_count": predicted.count("v21"),
            "v1_selected_count": predicted.count("v1"),
        })

router_tuning_df = pd.DataFrame(router_tuning_rows)
router_tuning_df.sort_values(
    ["router_accuracy", "v21_selected_count"],
    ascending=[False, False]
).head(10)

In [ ]:
# Cell 84: Select Best Router Threshold

best_router = router_tuning_df.sort_values(
    ["router_accuracy", "v21_selected_count"],
    ascending=[False, False]
).iloc[0].to_dict()

best_router

In [ ]:
# Cell 85: V3.1 Tuned Router

def choose_optimizer_v3_1(query: str) -> Dict[str, Any]:
    f = query_routing_features(query)

    semantic_risk_score = (
        0.35 * f["complexity_score"]
        + 0.25 * f["structural_signal"]
        + 0.20 * f["ppl_score"]
        + 0.20 * f["code_debug_signal"]
    )

    compression_opportunity_score = (
        0.45 * f["redundancy_ratio"]
        + 0.25 * min(f["token_len"] / 600, 1.0)
        + 0.15 * f["summarization_signal"]
        + 0.15 * f["creative_signal"]
    )

    risk_threshold = best_router["risk_threshold"]

    selected = "v1"
    reason = "Cost-first path selected."

    if semantic_risk_score >= risk_threshold:
        selected = "v21"
        reason = "Semantic risk exceeds tuned threshold; V2.1 selected."

    return {
        **f,
        "semantic_risk_score": round(semantic_risk_score, 4),
        "compression_opportunity_score": round(compression_opportunity_score, 4),
        "selected_optimizer": selected,
        "routing_reason": reason,
        "risk_threshold": risk_threshold,
    }

In [ ]:
# Cell 86: V3.1 Hybrid Call

def ask_groq_token_minimized_v3_1(query: str, model: str = MODEL) -> Dict[str, Any]:

    route = choose_optimizer_v3_1(query)

    if route["selected_optimizer"] == "v21":
        result = ask_groq_token_minimized_v2_1(query=query, model=model)
    else:
        result = ask_groq_token_minimized(query=query, model=model)

    result["v31_selected_optimizer"] = route["selected_optimizer"]
    result["v31_routing_reason"] = route["routing_reason"]
    result["semantic_risk_score"] = route["semantic_risk_score"]
    result["compression_opportunity_score"] = route["compression_opportunity_score"]
    result["risk_threshold"] = route["risk_threshold"]

    return result

In [ ]:
# Cell 87: Batch Evaluate V3.1

v31_rows = []

for use_case, query in test_queries.items():
    print(f"Running V3.1 evaluation: {use_case}")

    baseline = ask_groq_baseline(query)
    v1 = ask_groq_token_minimized(query)
    v21 = ask_groq_token_minimized_v2_1(query)
    v31 = ask_groq_token_minimized_v3_1(query)

    baseline_cost = cost_usd(baseline["input_tokens"], baseline["output_tokens"])
    v1_cost = cost_usd(v1["input_tokens"], v1["output_tokens"])
    v21_cost = cost_usd(v21["input_tokens"], v21["output_tokens"])
    v31_cost = cost_usd(v31["input_tokens"], v31["output_tokens"])

    v1_quality = semantic_quality_score(v1["answer"], query)
    v21_quality = semantic_quality_score(v21["answer"], query)
    v31_quality = semantic_quality_score(v31["answer"], query)

    cf = compression_factor(
        v31["original_est_tokens"],
        v31["optimized_est_tokens"]
    )

    tes = token_efficiency_score(
        information_density_value=v31["information_density"],
        quality_score=v31_quality,
        total_tokens=v31["total_tokens"]
    )

    v31_rows.append({
        "use_case": use_case,
        "selected_optimizer": v31["v31_selected_optimizer"],
        "routing_reason": v31["v31_routing_reason"],

        "v1_cost_reduction_percent": round(((baseline_cost - v1_cost) / baseline_cost) * 100, 2),
        "v21_cost_reduction_percent": round(((baseline_cost - v21_cost) / baseline_cost) * 100, 2),
        "v31_cost_reduction_percent": round(((baseline_cost - v31_cost) / baseline_cost) * 100, 2),

        "v1_quality": v1_quality,
        "v21_quality": v21_quality,
        "v31_quality": v31_quality,

        "v31_token_reduction_percent": round(
            ((baseline["total_tokens"] - v31["total_tokens"]) / baseline["total_tokens"]) * 100,
            2
        ),

        "semantic_risk_score": v31["semantic_risk_score"],
        "compression_opportunity_score": v31["compression_opportunity_score"],

        "v31_tes": tes,
        "v31_compression_factor": cf,
        "v31_total_tokens": v31["total_tokens"],
    })

v31_df = pd.DataFrame(v31_rows)
v31_df

In [ ]:
# Cell 88: Add GOES for V3.1

def add_goes_v31(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    df["tes_norm"] = df["v31_tes"].apply(
        lambda x: normalize_metric(x, df["v31_tes"].min(), df["v31_tes"].max())
    )

    df["cost_reduction_norm"] = df["v31_cost_reduction_percent"].apply(
        lambda x: normalize_metric(
            x,
            df["v31_cost_reduction_percent"].min(),
            df["v31_cost_reduction_percent"].max()
        )
    )

    df["compression_norm"] = df["v31_compression_factor"].apply(
        lambda x: normalize_metric(
            x,
            df["v31_compression_factor"].min(),
            df["v31_compression_factor"].max()
        )
    )

    df["v31_goes"] = df.apply(
        lambda r: global_optimization_efficiency_score(
            tes_norm=r["tes_norm"],
            cost_reduction_norm=r["cost_reduction_norm"],
            quality_score=r["v31_quality"],
            compression_norm=r["compression_norm"]
        ),
        axis=1
    )

    return df


v31_df = add_goes_v31(v31_df)
v31_df.sort_values("v31_goes", ascending=False)

In [ ]:
# Cell 89: Final V3.1 Summary

v31_summary = {
    "avg_v1_cost_reduction_percent": round(v31_df["v1_cost_reduction_percent"].mean(), 2),
    "avg_v21_cost_reduction_percent": round(v31_df["v21_cost_reduction_percent"].mean(), 2),
    "avg_v31_cost_reduction_percent": round(v31_df["v31_cost_reduction_percent"].mean(), 2),

    "avg_v1_quality": round(v31_df["v1_quality"].mean(), 4),
    "avg_v21_quality": round(v31_df["v21_quality"].mean(), 4),
    "avg_v31_quality": round(v31_df["v31_quality"].mean(), 4),

    "avg_v31_token_reduction_percent": round(v31_df["v31_token_reduction_percent"].mean(), 2),
    "avg_v31_goes": round(v31_df["v31_goes"].mean(), 4),

    "v31_selected_v1_count": int((v31_df["selected_optimizer"] == "v1").sum()),
    "v31_selected_v21_count": int((v31_df["selected_optimizer"] == "v21").sum()),
}

v31_summary

In [ ]:
# Cell 90: Human Interpretation

print("="*80)
print("FINAL V3.1 TUNED HYBRID ROUTER ANALYSIS")
print("="*80)

print("\nCost Reduction:")
print(f"V1   : {v31_summary['avg_v1_cost_reduction_percent']}%")
print(f"V2.1 : {v31_summary['avg_v21_cost_reduction_percent']}%")
print(f"V3.1 : {v31_summary['avg_v31_cost_reduction_percent']}%")

print("\nQuality:")
print(f"V1   : {v31_summary['avg_v1_quality']}")
print(f"V2.1 : {v31_summary['avg_v21_quality']}")
print(f"V3.1 : {v31_summary['avg_v31_quality']}")

print("\nGOES:")
print(f"V3.1 : {v31_summary['avg_v31_goes']}")

print("\nRouting Distribution:")
print(f"V1 selected   : {v31_summary['v31_selected_v1_count']} cases")
print(f"V2.1 selected : {v31_summary['v31_selected_v21_count']} cases")

print("\nVerdict:")

if (
    v31_summary["avg_v31_cost_reduction_percent"] >= v31_summary["avg_v1_cost_reduction_percent"]
    and v31_summary["avg_v31_quality"] >= v31_summary["avg_v1_quality"]
):
    print("✅ V3.1 beats V1 and is production-ready.")

elif (
    v31_summary["avg_v31_quality"] >= v31_summary["avg_v1_quality"]
    and v31_summary["avg_v31_cost_reduction_percent"] >= v31_summary["avg_v1_cost_reduction_percent"] - 3
):
    print("⚠️ V3.1 is a balanced production candidate.")

else:
    print("❌ V3.1 still needs tuning.")